In [1]:
import numpy as np

#Input: A list of integers
#Output: a permutation of those integers indexed by zero.
#Purpose: A wrapper function for making permutations from lists to handle both 
# the case of zero and non-zero indexed permutations.
def j_perm(li):
    li = list(li)
    if 0 in li:
        return Permutation(li, check_input = False)
    else:
        return Permutation([x-1 for x in li], check_input = False)
    return 

#Input: a permutation p and an integer n.
#Output: Create the fuzzy permutation matrix of p raised to size n
def genPermMatrix(p,n):
    k = len(p)
    if k==n:
        return matrix(QQ,n,n, lambda x,y : p[x] == y)
    return QQ(factorial(n-k))/QQ(binomial(n,k))*matrix(QQ,n,n,lambda x,y:sum([binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(k)]))
 

# first m rows of the above based on a k-permutation of [n]
def genPermMatrixByRow(p,n,m=None,k=None):
    if m==None:
        m=n
    l=len(p)
    if k==None:
        k=l
    if l==n:
        return matrix(QQ,m,n,lambda x,y:p[x]==y)
    return factorial(n-k)/binomial(n,k)*matrix(m,n,lambda x,y:sum([binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(l)]))

#Input: Two integers.
#Output: An inclusive list of all permutations between those two integers.
def genPermutationsBetween(min_length,max_length):
    cumulative_list = []
    for i in range(min_length,max_length+1,1):
        for p in Permutations(i):
            cumulative_list.append(j_perm(p))
    return cumulative_list

#Input: A list of combinations of permutations.
#Ouput: A copy of the input of lists containing only one element from each D4 equivalence class.
def reduceByD4(list_of_covers):
    d4_reduced_cover_set = []
    for cover1 in list_of_covers:
        is_symm = False
        for cover2 in d4_reduced_cover_set:
            if cover1 in cover2.d4Symmetries():
                is_symm = True
        if not is_symm:
            d4_reduced_cover_set.append(cover1)
    return d4_reduced_cover_set

#Input: A list of combinations of permutations
#Output: A nested list of those same combinations of permutations grouped by D4 equivalence.
def groupByD4(list_of_covers):
    d4_grouped_covers = [[cover] for cover in reduceByD4(list_of_covers)]
    for cover in list_of_covers:
        for group in d4_grouped_covers:
            if (cover in group[0].d4Symmetries()) and (cover not in group):
                group.append(cover)
    return d4_grouped_covers

threshold = 1e-12
def cleanHelper(x):
    return 0.0 if abs(x) < threshold else x

def cleanMatrix(M):
    return M.apply_map(cleanHelper)

def vanished(CoP,perm_list):
    if (CoP.isVanishing(CoP.maxTermLength())):
        return combination_of_permutations(CoP.get_coefficients(),CoP.get_terms())
    vec = CoP.make_vector(perm_list)
    vec[0] = -1/(ones.constantCoefficient(CoP.maxTermLength()))*(CoP.constantCoefficient(CoP.maxTermLength()))
    return combination_of_permutations.vector(vec,perm_list)

def vanished_vec(vec,perm_list):
    CoP = combination_of_permutations.vector(vec,perm_list)
    return vanished(CoP,perm_list).make_vector(perm_list)

#Input: A sage vector space
#Output: A basis for its orthogonal complement
def getOrthoCompBasis(V):
    # Get the basis matrix
    B = Matrix(QQ, V.basis())

    # Transpose B and compute its right kernel (null space of B^T)
    V_orth = B.transpose().right_kernel()

    # View basis of the orthogonal complement
    return V_orth.basis()


#A combination of permutations (CoP) represents a formal linear combination of permutations
#Input: A list of coefficients and a list of lists of the integers from 0 to k_i-1 or 1 to k_i. The integers k_i 
# are the lengths of each permutation.
class combination_of_permutations:
    def __init__(self, coefficients, terms):
            #Assumes coefficients are 1 if no coefficients are given.
            
            #Raises an error if the number of cofficients and terms are different. Otherwise the length of the 
            #constant cover s
            if (len(coefficients)!=len(terms)):
                raise ValueError("Coefficient and term lists must have the same length")

            self.coefficients = coefficients
            self.terms = [j_perm(term) for term in terms]
            
    @classmethod
    def nested(cls, nested_list):
        coefficient_list = [term_pair[0] for term_pair in nested_list]
        term_list = [term_pair[1] for term_pair in nested_list]
        return cls(coefficient_list,term_list)
    
    @classmethod
    def vector(cls, vector, permutation_list):
        coefficient_list = []
        term_list = []
        for i, entry in enumerate(vector.list()):
            if entry != 0:
                coefficient_list.append(entry)
                term_list.append(permutation_list[i])
        return cls(coefficient_list, term_list)
    
    @classmethod
    def latin_square(cls, latin):
        n = len(latin.row(0))
        coefficient_list = [1]*n
        term_list = []
        for i in range(n):
            term_list.append(j_perm(latin.row(i)))
        return cls(coefficient_list,term_list)
          
    #Returns a nested list representing the combination of permutations    
    def nested_list(self):
        nested_list = []
        for i in range(self.number_of_terms()):
            nested_list.append([self.coefficients[i], self.terms[i]])
        return nested_list
      
    def __iter__(self):
        return iter(self.nested_list())
    
    #Returns a nice string representation of the combination of permutations
    def __str__(self):
        nested_list = self.nested_list()
        if len(nested_list) == 0:
            return "0"
        nice_string = ""
        for term in nested_list:
            if term[0] == 0:
                continue
            if 0 in term[1]:
                term[1] = [x+1 for x in term[1]]
            if (term[0] > 0) and (term != nested_list[0]):
                nice_string += "+ "
            temp_string = ""
            if (term[1] != 1):
                temp_string += str(term[1])
            temp_string = temp_string[1:-1]
            temp_string = temp_string.replace(", ","")
            nice_string += str(term[0]) + "*(" + temp_string + ") "

        return nice_string
            
    #Returns a CoP whose coefficients have been multiplied by the scalar on the left. 
    #Syntax to use: s * CoP
    def __mul__(self, scalar):
        return combination_of_permutations([scalar*coefficient for coefficient in self.coefficients],self.terms)
    
    #Returns a CoP whose coefficients have been multiplied by the scalar on the right
    #Syntax to use: CoP * s
    def __rmul__(self, scalar):
        return combination_of_permutations([scalar*coefficient for coefficient in self.coefficients],self.terms)
    
    #Returns self + CoP
    #Syntax to use: self + CoP
    def __add__(self, CoP):
        #Create copies of the list, not references
        new_coeffs = self.coefficients[:]
        new_terms = self.terms[:]
        for i in range(CoP.number_of_terms()):
            current_term = CoP.get_terms()[i]
            current_coeff = CoP.get_coefficients()[i]
            if current_term in new_terms:
                new_coeffs[new_terms.index(current_term)] = new_coeffs[new_terms.index(current_term)] + current_coeff
                if new_coeffs[new_terms.index(current_term)] == 0:
                    deletion_index = new_terms.index(current_term)
                    del new_coeffs[deletion_index]
                    del new_terms[deletion_index]
            else:
                new_coeffs.append(current_coeff)
                new_terms.append(current_term)
        
        return combination_of_permutations(new_coeffs,new_terms)
     
    #Returns self - CoP
    #Syntax to use: self - Cop
    def __sub__(self,CoP):
        return self + -1*CoP
    
    #Returns True if self and CoP are the same linear combination
    #Syntax to use: self == CoP.
    def __eq__(self,CoP):
        coeffs1 = self.get_coefficients()
        coeffs2 = CoP.get_coefficients()
        if sorted(coeffs1) != sorted(coeffs2):
            return False
        
        terms1 = self.get_terms()
        terms2 = CoP.get_terms()
        
        if sorted(terms1) != sorted(terms2):
            return False
        
        return True
    
    def sort(self, rev=False):
        nested_list = self.nested_list()
        nested_list.sort(key = lambda term : (len(term[1]),str(term[1])), reverse = rev)
        CoP = combination_of_permutations.nested(nested_list)
        self.coefficients = CoP.get_coefficients()
        self.terms = CoP.get_terms()
    
    #Returns the list of coefficients of the combination of permutations
    def get_coefficients(self):
        return self.coefficients
    
    #Returns the list of terms of the combination of permutations
    def get_terms(self):
        return self.terms
            
    #Return the number of terms of the combination of permutations
    def number_of_terms(self):
        return len(self.get_terms())
    
    def maxTermLength(self):
        return len(sorted(self.get_terms(), key = lambda x: len(x))[-1])
        
    #Returns a string formatted for Latex.
    def dispLatex(self):
        nested_list = self.nested_list()
        if len(nested_list) == 0:
            return "\\item $\\rho = 0 $"
        nice_string = ""
        for term in nested_list:
            if 0 in term[1]:
                term[1] = [x+1 for x in term[1]]
            if (term[0] > 0) and (term != nested_list[0]):
                nice_string += "+"
            temp_string = ""
            temp_string += str(term[1])
            temp_string = temp_string[1:-1]
            temp_string = temp_string.replace(", ","")
            if str(term[0]) == '1':
                nice_string += "\\perm{" + temp_string + "} "
            else:
                nice_string += str(term[0]) + "(\\perm{" + temp_string + "}) "
        nice_string = "\\item $\\rho=" + nice_string + "$" 
        
        return nice_string
    
    #Returns the list of combinations of permutations created by self under D4.
    def d4Symmetries(self, remove_duplicates = True):
        terms = self.terms
        symmetry_array = []
        solution_list = []
        for p in terms:
            n=len(p)
            ans=[p] #Identity operation
            ans+=[[n-1-p[i] for i in range(n)]] 
            ans+=[[p[n-1-i] for i in range(n)]]
            ans+=[[n-1-p[n-1-i] for i in range(n)]]
            
            q=[0]*n
            for i in range(n):
                q[p[i]]=i
            ans+=[q]
            ans+=[[n-1-q[i] for i in range(n)]]
            ans+=[[q[n-1-i] for i in range(n)]]
            ans+=[[n-1-q[n-1-i] for i in range(n)]]
            symmetry_array.append(ans)
        for j in range(8):
            acted_on_terms = []
            for i in range(len(terms)):
                acted_on_terms.append(symmetry_array[i][j])
            rho = combination_of_permutations(self.coefficients,acted_on_terms)
            if remove_duplicates:
                if rho not in solution_list:
                    solution_list.append(rho)
            else:
                 solution_list.append(rho)   
        return solution_list
    
    #Returns a sage vector whose entries are the coefficients corresponding to those elements of the permutation list.
    def make_vector(self, permutation_list):
        v = np.zeros(len(permutation_list))
        for i, term in enumerate(self.terms):
            v[permutation_list.index(term)] = self.coefficients[i]
        vec =vector(QQ, v)
        return vec
    
    def genFuzzyMatrixList(self,n):
        list_of_matrices = []
        for i, p in enumerate(self.terms):    
            k = len(p)
            if k==n:
                m = (matrix(QQ,n,n, lambda x,y : int(p[x] == y)))
            else:
                m = factorial(n-k)/binomial(n,k)*matrix(QQ,n,n,lambda x,y:sum([binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(k)]))
            list_of_matrices.append(self.coefficients[i]*m)
        return list_of_matrices
    
    def genFuzzyMatrix(self,n):
        fuzzies = self.genFuzzyMatrixList(n)
        fuzzy_matrix = Matrix(QQ,fuzzies[0])
        for m in fuzzies[1:]:
            fuzzy_matrix = Matrix(QQ,fuzzy_matrix + m)
        return fuzzy_matrix
    
    def isConstant(self,n):
        fuzzy = self.genFuzzyMatrix(n)
        constant = fuzzy[0][0]
        fuzzy = fuzzy.apply_map(lambda x: x - constant)
        fuzzy = cleanMatrix(fuzzy)
        return Matrix(QQ,fuzzy).is_zero()
    
    def constantCoefficient(self,n):
        if not self.isConstant(n):
            raise ValueError
        else:
            return self.genFuzzyMatrix(n)[0][0]        
        
    def isVanishing(self,n):
        return(Matrix(QQ,self.genFuzzyMatrix(n)).is_zero())
    
    #Redundent with maxTermLength, must choose one. This one is coded better
    def order(self):
        return(max([len(term) for term in self.terms]))
    
    
    #Input: Nothing
    #Ouput: Returns a graph with the terms of the CoP as vertices, with edges where two permutations have a common entry
    def compGraph(self):
        order = self.order()
        termList = self.get_terms()
        for term in termList:
            if len(term) != order:
                return None
        adjMat = zero_matrix(QQ,len(termList),len(termList))
        for i in range(len(termList)):
            for j in range(len(termList)):
                if i == j:
                    continue
                if 0 in list((vector(QQ,list(termList[i])) - vector(QQ,list(termList[j])))): 
                    adjMat[i,j] = 1
        g = Graph(adjMat, format='adjacency_matrix')
        return g
    
    
    #Input: Nothing
    #Ouput: Returns a weighted graph with the terms of the CoP as vertices, and the weights as the number of common entries.
    def weightedCompGraph(self):
        order = self.order()
        termList = self.get_terms()
        for term in termList:
            if len(term) != order:
                return None
        adjMat = zero_matrix(QQ,len(termList),len(termList))
        for i in range(len(termList)):
            for j in range(len(termList)):
                if i == j:
                    continue
                adjMat[i,j] = list((vector(QQ,list(termList[i])) - vector(QQ,list(termList[j])))).count(0)
        g = Graph(adjMat, format='weighted_adjacency_matrix')
        return g
        
# TODO
# Cycle notation constructor.

In [4]:
most_repeated_entry_count = {
    (3,2) : (5, [[1, 2], [2, 1]]),
    (3,3) : (3, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]),
    (4,2) : (4, [[1, 2], [2, 1]]),
    (4,3) : (4, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]),
    (4,4) : (4, [[1, 2, 3, 4], [1, 2, 4, 3], [1, 3, 2, 4], [1, 3, 4, 2], [1, 4, 2, 3], [1, 4, 3, 2], [2, 1, 3, 4], [2, 1, 4, 3], [2, 3, 1, 4], [2, 3, 4, 1], [2, 4, 1, 3], [2, 4, 3, 1], [3, 1, 2, 4], [3, 1, 4, 2], [3, 2, 1, 4], [3, 2, 4, 1], [3, 4, 1, 2], [3, 4, 2, 1], [4, 1, 2, 3], [4, 1, 3, 2], [4, 2, 1, 3], [4, 2, 3, 1], [4, 3, 1, 2], [4, 3, 2, 1]]),
    (5,2) : (9, [[1, 2], [2, 1]]),
    (5,3) : (7, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]),
    (5,4) : (4, [[1, 2, 3, 4], [1, 3, 2, 4], [1, 4, 3, 2], [2, 1, 4, 3], [2, 3, 4, 1], [2, 4, 1, 3], [3, 1, 4, 2], [3, 2, 1, 4], [3, 4, 1, 2], [4, 1, 2, 3], [4, 2, 3, 1], [4, 3, 2, 1]]),
    (5,5) : (5, [[1, 2, 3, 4, 5], [1, 2, 3, 5, 4], [1, 2, 4, 3, 5], [1, 2, 4, 5, 3], [1, 2, 5, 3, 4], [1, 2, 5, 4, 3], [1, 3, 2, 4, 5], [1, 3, 2, 5, 4], [1, 3, 4, 2, 5], [1, 3, 4, 5, 2], [1, 3, 5, 2, 4], [1, 3, 5, 4, 2], [1, 4, 2, 3, 5], [1, 4, 2, 5, 3], [1, 4, 3, 2, 5], [1, 4, 3, 5, 2], [1, 4, 5, 2, 3], [1, 4, 5, 3, 2], [1, 5, 2, 3, 4], [1, 5, 2, 4, 3], [1, 5, 3, 2, 4], [1, 5, 3, 4, 2], [1, 5, 4, 2, 3], [1, 5, 4, 3, 2], [2, 1, 3, 4, 5], [2, 1, 3, 5, 4], [2, 1, 4, 3, 5], [2, 1, 4, 5, 3], [2, 1, 5, 3, 4], [2, 1, 5, 4, 3], [2, 3, 1, 4, 5], [2, 3, 1, 5, 4], [2, 3, 4, 1, 5], [2, 3, 4, 5, 1], [2, 3, 5, 1, 4], [2, 3, 5, 4, 1], [2, 4, 1, 3, 5], [2, 4, 1, 5, 3], [2, 4, 3, 1, 5], [2, 4, 3, 5, 1], [2, 4, 5, 1, 3], [2, 4, 5, 3, 1], [2, 5, 1, 3, 4], [2, 5, 1, 4, 3], [2, 5, 3, 1, 4], [2, 5, 3, 4, 1], [2, 5, 4, 1, 3], [2, 5, 4, 3, 1], [3, 1, 2, 4, 5], [3, 1, 2, 5, 4], [3, 1, 4, 2, 5], [3, 1, 4, 5, 2], [3, 1, 5, 2, 4], [3, 1, 5, 4, 2], [3, 2, 1, 4, 5], [3, 2, 1, 5, 4], [3, 2, 4, 1, 5], [3, 2, 4, 5, 1], [3, 2, 5, 1, 4], [3, 2, 5, 4, 1], [3, 4, 1, 2, 5], [3, 4, 1, 5, 2], [3, 4, 2, 1, 5], [3, 4, 2, 5, 1], [3, 4, 5, 1, 2], [3, 4, 5, 2, 1], [3, 5, 1, 2, 4], [3, 5, 1, 4, 2], [3, 5, 2, 1, 4], [3, 5, 2, 4, 1], [3, 5, 4, 1, 2], [3, 5, 4, 2, 1], [4, 1, 2, 3, 5], [4, 1, 2, 5, 3], [4, 1, 3, 2, 5], [4, 1, 3, 5, 2], [4, 1, 5, 2, 3], [4, 1, 5, 3, 2], [4, 2, 1, 3, 5], [4, 2, 1, 5, 3], [4, 2, 3, 1, 5], [4, 2, 3, 5, 1], [4, 2, 5, 1, 3], [4, 2, 5, 3, 1], [4, 3, 1, 2, 5], [4, 3, 1, 5, 2], [4, 3, 2, 1, 5], [4, 3, 2, 5, 1], [4, 3, 5, 1, 2], [4, 3, 5, 2, 1], [4, 5, 1, 2, 3], [4, 5, 1, 3, 2], [4, 5, 2, 1, 3], [4, 5, 2, 3, 1], [4, 5, 3, 1, 2], [4, 5, 3, 2, 1], [5, 1, 2, 3, 4], [5, 1, 2, 4, 3], [5, 1, 3, 2, 4], [5, 1, 3, 4, 2], [5, 1, 4, 2, 3], [5, 1, 4, 3, 2], [5, 2, 1, 3, 4], [5, 2, 1, 4, 3], [5, 2, 3, 1, 4], [5, 2, 3, 4, 1], [5, 2, 4, 1, 3], [5, 2, 4, 3, 1], [5, 3, 1, 2, 4], [5, 3, 1, 4, 2], [5, 3, 2, 1, 4], [5, 3, 2, 4, 1], [5, 3, 4, 1, 2], [5, 3, 4, 2, 1], [5, 4, 1, 2, 3], [5, 4, 1, 3, 2], [5, 4, 2, 1, 3], [5, 4, 2, 3, 1], [5, 4, 3, 1, 2], [5, 4, 3, 2, 1]]),
    (6,2) : (4, [[1, 2], [2, 1]]),
    (6,3) : (8, [[1, 2, 3], [3, 2, 1]]),
    (6,4) : (8, [[1, 2, 3, 4], [2, 1, 4, 3], [2, 4, 1, 3], [3, 1, 4, 2], [3, 4, 1, 2], [4, 3, 2, 1]]),
    (6,5) : (5, [[1, 2, 4, 3, 5], [1, 3, 2, 4, 5], [1, 3, 2, 5, 4], [1, 3, 4, 2, 5], [1, 4, 2, 3, 5], [2, 1, 4, 3, 5], [4, 5, 2, 3, 1], [5, 2, 4, 3, 1], [5, 3, 2, 4, 1], [5, 3, 4, 1, 2], [5, 3, 4, 2, 1], [5, 4, 2, 3, 1]]),
    
    (3,2,2) : (9, [([1, 2], [2, 1]), ([2, 1], [1, 2])]),
    (3,3,2) : (6, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])]),
    (4,2,2) : (16, [([1, 2], [2, 1]), ([2, 1], [1, 2])]),
    (4,3,2) : (8, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])]),
    (4,3,3) : (8, [([1, 2, 3], [3, 2, 1]), ([3, 2, 1], [1, 2, 3])]),
    (4,4,2) : (8, [([2, 1, 4, 3], [1, 2]), ([2, 1, 4, 3], [2, 1]), ([3, 4, 1, 2], [1, 2]), ([3, 4, 1, 2], [2, 1])]),
    (4,4,3) : (8, [([2, 1, 4, 3], [3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3])]),
    (5,2,2) : (25, [([1, 2], [2, 1]), ([2, 1], [1, 2])]),
    (5,3,2) : (9, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([1, 3, 2], [1, 2]), ([1, 3, 2], [2, 1]), ([2, 1, 3], [1, 2]), ([2, 1, 3], [2, 1]), ([2, 3, 1], [1, 2]), ([2, 3, 1], [2, 1]), ([3, 1, 2], [1, 2]), ([3, 1, 2], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])]),
    (5,3,3) : (9, [([1, 2, 3], [2, 3, 1]), ([1, 2, 3], [3, 1, 2]), ([1, 3, 2], [3, 2, 1]), ([2, 1, 3], [3, 2, 1]), ([2, 3, 1], [1, 2, 3]), ([3, 1, 2], [1, 2, 3]), ([3, 2, 1], [1, 3, 2]), ([3, 2, 1], [2, 1, 3])]),
    (5,4,2) : (10, [([2, 1, 4, 3], [2, 1]), ([3, 4, 1, 2], [1, 2])]),
    (5,4,3) : (17, [([2, 1, 4, 3], [3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3])]),
    (5,4,4) : (9, [([1, 2, 3, 4], [2, 1, 4, 3]), ([1, 2, 3, 4], [2, 4, 1, 3]), ([1, 2, 3, 4], [3, 1, 4, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 4, 3], [1, 4, 3, 2]), ([1, 2, 4, 3], [2, 1, 3, 4]), ([1, 3, 2, 4], [2, 1, 4, 3]), ([1, 3, 2, 4], [2, 4, 1, 3]), ([1, 3, 2, 4], [3, 1, 4, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 4, 2], [3, 1, 2, 4]), ([1, 4, 2, 3], [2, 3, 1, 4]), ([1, 4, 3, 2], [1, 2, 4, 3]), ([1, 4, 3, 2], [3, 2, 1, 4]), ([2, 1, 3, 4], [1, 2, 4, 3]), ([2, 1, 3, 4], [3, 2, 1, 4]), ([2, 1, 4, 3], [1, 2, 3, 4]), ([2, 1, 4, 3], [1, 3, 2, 4]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 3, 1, 4], [1, 4, 2, 3]), ([2, 3, 4, 1], [3, 4, 2, 1]), ([2, 3, 4, 1], [4, 1, 2, 3]), ([2, 4, 1, 3], [1, 2, 3, 4]), ([2, 4, 1, 3], [1, 3, 2, 4]), ([2, 4, 1, 3], [4, 2, 3, 1]), ([2, 4, 1, 3], [4, 3, 2, 1]), ([2, 4, 3, 1], [4, 2, 1, 3]), ([3, 1, 2, 4], [1, 3, 4, 2]), ([3, 1, 4, 2], [1, 2, 3, 4]), ([3, 1, 4, 2], [1, 3, 2, 4]), ([3, 1, 4, 2], [4, 2, 3, 1]), ([3, 1, 4, 2], [4, 3, 2, 1]), ([3, 2, 1, 4], [1, 4, 3, 2]), ([3, 2, 1, 4], [2, 1, 3, 4]), ([3, 2, 4, 1], [4, 1, 3, 2]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 3, 2, 4]), ([3, 4, 1, 2], [4, 2, 3, 1]), ([3, 4, 1, 2], [4, 3, 2, 1]), ([3, 4, 2, 1], [2, 3, 4, 1]), ([3, 4, 2, 1], [4, 3, 1, 2]), ([4, 1, 2, 3], [2, 3, 4, 1]), ([4, 1, 2, 3], [4, 3, 1, 2]), ([4, 1, 3, 2], [3, 2, 4, 1]), ([4, 2, 1, 3], [2, 4, 3, 1]), ([4, 2, 3, 1], [2, 1, 4, 3]), ([4, 2, 3, 1], [2, 4, 1, 3]), ([4, 2, 3, 1], [3, 1, 4, 2]), ([4, 2, 3, 1], [3, 4, 1, 2]), ([4, 3, 1, 2], [3, 4, 2, 1]), ([4, 3, 1, 2], [4, 1, 2, 3]), ([4, 3, 2, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [2, 4, 1, 3]), ([4, 3, 2, 1], [3, 1, 4, 2]), ([4, 3, 2, 1], [3, 4, 1, 2])]),
    (5,5,2) : (12, [([2, 1, 3, 5, 4], [1, 2]), ([2, 1, 3, 5, 4], [1, 2]), ([2, 1, 3, 5, 4], [2, 1]), ([4, 5, 3, 1, 2], [1, 2]), ([4, 5, 3, 1, 2], [2, 1]), ([4, 5, 3, 1, 2], [2, 1])]),
    (5,5,3) : (10, [([1, 2, 3, 4, 5], [2, 3, 1]), ([1, 2, 3, 4, 5], [3, 1, 2]), ([1, 3, 2, 5, 4], [3, 2, 1]), ([1, 4, 3, 2, 5], [1, 3, 2]), ([1, 4, 3, 2, 5], [2, 1, 3]), ([1, 5, 3, 4, 2], [2, 1, 3]), ([1, 5, 4, 3, 2], [2, 1, 3]), ([2, 1, 3, 5, 4], [3, 2, 1]), ([2, 1, 4, 3, 5], [3, 2, 1]), ([2, 1, 4, 5, 3], [3, 2, 1]), ([2, 1, 5, 3, 4], [3, 2, 1]), ([2, 3, 1, 5, 4], [3, 2, 1]), ([2, 3, 4, 5, 1], [3, 1, 2]), ([2, 4, 3, 5, 1], [3, 1, 2]), ([3, 1, 2, 5, 4], [3, 2, 1]), ([3, 5, 4, 1, 2], [1, 2, 3]), ([4, 2, 3, 1, 5], [1, 3, 2]), ([4, 3, 2, 1, 5], [1, 3, 2]), ([4, 3, 5, 1, 2], [1, 2, 3]), ([4, 5, 1, 3, 2], [1, 2, 3]), ([4, 5, 2, 1, 3], [1, 2, 3]), ([4, 5, 2, 3, 1], [1, 2, 3]), ([4, 5, 3, 1, 2], [1, 2, 3]), ([5, 1, 2, 3, 4], [2, 3, 1]), ([5, 1, 3, 2, 4], [2, 3, 1]), ([5, 2, 3, 4, 1], [2, 3, 1]), ([5, 2, 3, 4, 1], [3, 1, 2]), ([5, 3, 4, 1, 2], [1, 2, 3]), ([5, 4, 3, 2, 1], [1, 3, 2]), ([5, 4, 3, 2, 1], [2, 1, 3])]),
    (5,5,4) : (9, [([1, 2, 3, 4, 5], [3, 4, 1, 2]), ([1, 2, 3, 4, 5], [3, 4, 1, 2]), ([1, 2, 3, 4, 5], [3, 4, 1, 2]), ([1, 2, 3, 5, 4], [3, 4, 1, 2]), ([1, 2, 3, 5, 4], [3, 4, 1, 2]), ([1, 2, 3, 5, 4], [3, 4, 1, 2]), ([1, 2, 5, 4, 3], [4, 3, 2, 1]), ([1, 2, 5, 4, 3], [4, 3, 2, 1]), ([1, 3, 4, 5, 2], [4, 1, 2, 3]), ([1, 4, 3, 2, 5], [2, 1, 4, 3]), ([1, 4, 3, 2, 5], [2, 1, 4, 3]), ([1, 4, 3, 2, 5], [2, 1, 4, 3]), ([1, 5, 2, 3, 4], [2, 3, 4, 1]), ([1, 5, 4, 3, 2], [3, 2, 1, 4]), ([2, 1, 3, 4, 5], [3, 4, 1, 2]), ([2, 1, 3, 4, 5], [3, 4, 1, 2]), ([2, 1, 3, 4, 5], [3, 4, 1, 2]), ([2, 1, 3, 5, 4], [3, 4, 1, 2]), ([2, 1, 3, 5, 4], [3, 4, 1, 2]), ([2, 1, 3, 5, 4], [3, 4, 1, 2]), ([2, 1, 5, 4, 3], [4, 3, 2, 1]), ([2, 1, 5, 4, 3], [4, 3, 2, 1]), ([2, 3, 4, 1, 5], [4, 1, 2, 3]), ([2, 3, 4, 5, 1], [4, 1, 2, 3]), ([2, 5, 3, 1, 4], [3, 1, 4, 2]), ([2, 5, 3, 1, 4], [3, 1, 4, 2]), ([2, 5, 3, 1, 4], [3, 1, 4, 2]), ([2, 5, 3, 1, 4], [3, 1, 4, 2]), ([2, 5, 4, 3, 1], [3, 2, 1, 4]), ([3, 2, 1, 4, 5], [4, 3, 2, 1]), ([3, 2, 1, 4, 5], [4, 3, 2, 1]), ([3, 2, 1, 5, 4], [4, 3, 2, 1]), ([3, 2, 1, 5, 4], [4, 3, 2, 1]), ([3, 4, 5, 1, 2], [1, 2, 3, 4]), ([3, 4, 5, 1, 2], [1, 2, 3, 4]), ([3, 4, 5, 2, 1], [1, 2, 3, 4]), ([3, 4, 5, 2, 1], [1, 2, 3, 4]), ([4, 1, 2, 3, 5], [2, 3, 4, 1]), ([4, 1, 3, 5, 2], [2, 4, 1, 3]), ([4, 1, 3, 5, 2], [2, 4, 1, 3]), ([4, 1, 3, 5, 2], [2, 4, 1, 3]), ([4, 1, 3, 5, 2], [2, 4, 1, 3]), ([4, 3, 2, 1, 5], [1, 4, 3, 2]), ([4, 3, 2, 5, 1], [1, 4, 3, 2]), ([4, 5, 1, 2, 3], [1, 2, 3, 4]), ([4, 5, 1, 2, 3], [1, 2, 3, 4]), ([4, 5, 3, 1, 2], [2, 1, 4, 3]), ([4, 5, 3, 1, 2], [2, 1, 4, 3]), ([4, 5, 3, 1, 2], [2, 1, 4, 3]), ([4, 5, 3, 2, 1], [2, 1, 4, 3]), ([4, 5, 3, 2, 1], [2, 1, 4, 3]), ([4, 5, 3, 2, 1], [2, 1, 4, 3]), ([5, 1, 2, 3, 4], [2, 3, 4, 1]), ([5, 1, 4, 3, 2], [3, 2, 1, 4]), ([5, 2, 3, 4, 1], [3, 4, 1, 2]), ([5, 2, 3, 4, 1], [3, 4, 1, 2]), ([5, 2, 3, 4, 1], [3, 4, 1, 2]), ([5, 3, 2, 1, 4], [1, 4, 3, 2]), ([5, 4, 1, 2, 3], [1, 2, 3, 4]), ([5, 4, 1, 2, 3], [1, 2, 3, 4]), ([5, 4, 3, 1, 2], [2, 1, 4, 3]), ([5, 4, 3, 1, 2], [2, 1, 4, 3]), ([5, 4, 3, 1, 2], [2, 1, 4, 3]), ([5, 4, 3, 2, 1], [2, 1, 4, 3]), ([5, 4, 3, 2, 1], [2, 1, 4, 3]), ([5, 4, 3, 2, 1], [2, 1, 4, 3])]),    (6,2,2) : (36, [([1, 2], [2, 1]), ([2, 1], [1, 2])]),
    (6,3,2) : (10, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])]),
    (6,3,3) : (10, [([1, 2, 3], [3, 2, 1]), ([3, 2, 1], [1, 2, 3])]),
    (6,4,2) : (12, [([2, 1, 4, 3], [1, 2]), ([2, 1, 4, 3], [2, 1]), ([3, 4, 1, 2], [1, 2]), ([3, 4, 1, 2], [2, 1])]),
    (6,4,3) : (18, [([2, 1, 4, 3], [3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3])]),
    (6,4,4) : (12, [([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [4, 2, 3, 1]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3, 4, 1, 2], [1, 3, 2, 4]), ([3, 4, 1, 2], [1, 3, 2, 4]), ([3, 4, 1, 2], [1, 3, 2, 4]), ([3, 4, 1, 2], [1, 3, 2, 4]), ([4, 2, 3, 1], [1, 2, 3, 4]), ([4, 2, 3, 1], [2, 1, 4, 3]), ([4, 2, 3, 1], [2, 1, 4, 3]), ([4, 2, 3, 1], [2, 1, 4, 3]), ([4, 2, 3, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [1, 3, 2, 4]), ([4, 3, 2, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [2, 1, 4, 3]), ([4, 3, 2, 1], [2, 1, 4, 3])]),
    (6,5,2) : (12, [([2, 1, 3, 5, 4], [2, 1]), ([4, 5, 3, 1, 2], [1, 2])]),
    (6,5,3) : (12, [([1, 2, 3, 4, 5], [1, 2, 3]), ([1, 4, 3, 2, 5], [3, 2, 1]), ([5, 2, 3, 4, 1], [1, 2, 3]), ([5, 4, 3, 2, 1], [3, 2, 1])]),
    (6,5,4) : (16, [([2, 1, 3, 5, 4], [3, 4, 1, 2]), ([2, 5, 3, 1, 4], [3, 1, 4, 2]), ([4, 1, 3, 5, 2], [2, 4, 1, 3]), ([4, 5, 3, 1, 2], [2, 1, 4, 3])]),
    (6,5,5) : (9, [([1, 2, 4, 3, 5], [5, 2, 4, 3, 1]), ([1, 2, 4, 3, 5], [5, 2, 4, 3, 1]), ([1, 3, 2, 4, 5], [5, 3, 2, 4, 1]), ([1, 3, 2, 4, 5], [5, 3, 2, 4, 1]), ([1, 3, 2, 5, 4], [5, 3, 2, 4, 1]), ([1, 3, 4, 2, 5], [5, 3, 4, 1, 2]), ([1, 3, 4, 2, 5], [5, 3, 4, 2, 1]), ([1, 3, 4, 2, 5], [5, 3, 4, 2, 1]), ([1, 4, 2, 3, 5], [4, 5, 2, 3, 1]), ([1, 4, 2, 3, 5], [5, 4, 2, 3, 1]), ([1, 4, 2, 3, 5], [5, 4, 2, 3, 1]), ([2, 1, 4, 3, 5], [5, 2, 4, 3, 1]), ([4, 5, 2, 3, 1], [1, 4, 2, 3, 5]), ([5, 2, 4, 3, 1], [1, 2, 4, 3, 5]), ([5, 2, 4, 3, 1], [1, 2, 4, 3, 5]), ([5, 2, 4, 3, 1], [2, 1, 4, 3, 5]), ([5, 3, 2, 4, 1], [1, 3, 2, 4, 5]), ([5, 3, 2, 4, 1], [1, 3, 2, 4, 5]), ([5, 3, 2, 4, 1], [1, 3, 2, 5, 4]), ([5, 3, 4, 1, 2], [1, 3, 4, 2, 5]), ([5, 3, 4, 2, 1], [1, 3, 4, 2, 5]), ([5, 3, 4, 2, 1], [1, 3, 4, 2, 5]), ([5, 4, 2, 3, 1], [1, 4, 2, 3, 5]), ([5, 4, 2, 3, 1], [1, 4, 2, 3, 5])]),
    (6,6,2) : (8, [([1, 3, 2, 5, 4, 6], [1, 2]), ([1, 3, 2, 5, 4, 6], [1, 2]), ([1, 3, 2, 5, 4, 6], [1, 2]), ([1, 3, 2, 5, 4, 6], [2, 1]), ([1, 3, 2, 5, 4, 6], [2, 1]), ([1, 3, 2, 5, 4, 6], [2, 1]), ([1, 3, 2, 5, 4, 6], [2, 1]), ([1, 4, 5, 2, 3, 6], [1, 2]), ([1, 4, 5, 2, 3, 6], [1, 2]), ([1, 4, 5, 2, 3, 6], [1, 2]), ([1, 4, 5, 2, 3, 6], [1, 2]), ([1, 4, 5, 2, 3, 6], [2, 1]), ([1, 4, 5, 2, 3, 6], [2, 1]), ([1, 4, 5, 2, 3, 6], [2, 1]), ([2, 1, 3, 4, 6, 5], [1, 2]), ([2, 1, 3, 4, 6, 5], [1, 2]), ([2, 1, 3, 4, 6, 5], [1, 2]), ([2, 1, 3, 4, 6, 5], [1, 2]), ([2, 1, 3, 4, 6, 5], [2, 1]), ([2, 1, 3, 4, 6, 5], [2, 1]), ([2, 1, 3, 4, 6, 5], [2, 1]), ([2, 1, 3, 4, 6, 5], [2, 1]), ([2, 1, 3, 4, 6, 5], [2, 1]), ([2, 1, 4, 3, 6, 5], [1, 2]), ([2, 1, 4, 3, 6, 5], [1, 2]), ([2, 1, 4, 3, 6, 5], [1, 2]), ([2, 1, 4, 3, 6, 5], [1, 2]), ([2, 1, 4, 3, 6, 5], [2, 1]), ([2, 1, 4, 3, 6, 5], [2, 1]), ([2, 1, 4, 3, 6, 5], [2, 1]), ([2, 1, 4, 3, 6, 5], [2, 1]), ([2, 1, 4, 3, 6, 5], [2, 1]), ([3, 2, 1, 6, 5, 4], [1, 2]), ([3, 2, 1, 6, 5, 4], [1, 2]), ([3, 2, 1, 6, 5, 4], [1, 2]), ([3, 2, 1, 6, 5, 4], [1, 2]), ([3, 2, 1, 6, 5, 4], [1, 2]), ([3, 2, 1, 6, 5, 4], [2, 1]), ([3, 2, 1, 6, 5, 4], [2, 1]), ([3, 2, 1, 6, 5, 4], [2, 1]), ([3, 2, 1, 6, 5, 4], [2, 1]), ([3, 2, 1, 6, 5, 4], [2, 1]), ([3, 5, 1, 6, 2, 4], [1, 2]), ([3, 5, 1, 6, 2, 4], [1, 2]), ([3, 5, 1, 6, 2, 4], [1, 2]), ([3, 5, 1, 6, 2, 4], [1, 2]), ([3, 5, 1, 6, 2, 4], [1, 2]), ([3, 5, 1, 6, 2, 4], [2, 1]), ([3, 5, 1, 6, 2, 4], [2, 1]), ([3, 5, 1, 6, 2, 4], [2, 1]), ([3, 5, 1, 6, 2, 4], [2, 1]), ([3, 5, 1, 6, 2, 4], [2, 1]), ([4, 2, 6, 1, 5, 3], [1, 2]), ([4, 2, 6, 1, 5, 3], [1, 2]), ([4, 2, 6, 1, 5, 3], [1, 2]), ([4, 2, 6, 1, 5, 3], [1, 2]), ([4, 2, 6, 1, 5, 3], [1, 2]), ([4, 2, 6, 1, 5, 3], [2, 1]), ([4, 2, 6, 1, 5, 3], [2, 1]), ([4, 2, 6, 1, 5, 3], [2, 1]), ([4, 2, 6, 1, 5, 3], [2, 1]), ([4, 2, 6, 1, 5, 3], [2, 1]), ([4, 5, 6, 1, 2, 3], [1, 2]), ([4, 5, 6, 1, 2, 3], [1, 2]), ([4, 5, 6, 1, 2, 3], [1, 2]), ([4, 5, 6, 1, 2, 3], [1, 2]), ([4, 5, 6, 1, 2, 3], [1, 2]), ([4, 5, 6, 1, 2, 3], [2, 1]), ([4, 5, 6, 1, 2, 3], [2, 1]), ([4, 5, 6, 1, 2, 3], [2, 1]), ([4, 5, 6, 1, 2, 3], [2, 1]), ([4, 5, 6, 1, 2, 3], [2, 1]), ([5, 6, 3, 4, 1, 2], [1, 2]), ([5, 6, 3, 4, 1, 2], [1, 2]), ([5, 6, 3, 4, 1, 2], [1, 2]), ([5, 6, 3, 4, 1, 2], [1, 2]), ([5, 6, 3, 4, 1, 2], [1, 2]), ([5, 6, 3, 4, 1, 2], [2, 1]), ([5, 6, 3, 4, 1, 2], [2, 1]), ([5, 6, 3, 4, 1, 2], [2, 1]), ([5, 6, 3, 4, 1, 2], [2, 1]), ([5, 6, 4, 3, 1, 2], [1, 2]), ([5, 6, 4, 3, 1, 2], [1, 2]), ([5, 6, 4, 3, 1, 2], [1, 2]), ([5, 6, 4, 3, 1, 2], [1, 2]), ([5, 6, 4, 3, 1, 2], [1, 2]), ([5, 6, 4, 3, 1, 2], [2, 1]), ([5, 6, 4, 3, 1, 2], [2, 1]), ([5, 6, 4, 3, 1, 2], [2, 1]), ([5, 6, 4, 3, 1, 2], [2, 1]), ([6, 3, 2, 5, 4, 1], [1, 2]), ([6, 3, 2, 5, 4, 1], [1, 2]), ([6, 3, 2, 5, 4, 1], [1, 2]), ([6, 3, 2, 5, 4, 1], [2, 1]), ([6, 3, 2, 5, 4, 1], [2, 1]), ([6, 3, 2, 5, 4, 1], [2, 1]), ([6, 3, 2, 5, 4, 1], [2, 1]), ([6, 4, 5, 2, 3, 1], [1, 2]), ([6, 4, 5, 2, 3, 1], [1, 2]), ([6, 4, 5, 2, 3, 1], [1, 2]), ([6, 4, 5, 2, 3, 1], [1, 2]), ([6, 4, 5, 2, 3, 1], [2, 1]), ([6, 4, 5, 2, 3, 1], [2, 1]), ([6, 4, 5, 2, 3, 1], [2, 1])]),
    (6,6,3) : (12, [([1, 3, 2, 5, 4, 6], [1, 2, 3]), ([1, 4, 5, 2, 3, 6], [3, 2, 1]), ([2, 1, 3, 4, 6, 5], [1, 2, 3]), ([2, 1, 3, 4, 6, 5], [3, 2, 1]), ([2, 1, 4, 3, 6, 5], [1, 2, 3]), ([2, 1, 4, 3, 6, 5], [3, 2, 1]), ([3, 2, 1, 6, 5, 4], [3, 2, 1]), ([3, 5, 1, 6, 2, 4], [3, 2, 1]), ([4, 2, 6, 1, 5, 3], [1, 2, 3]), ([4, 5, 6, 1, 2, 3], [1, 2, 3]), ([5, 6, 3, 4, 1, 2], [1, 2, 3]), ([5, 6, 3, 4, 1, 2], [3, 2, 1]), ([5, 6, 4, 3, 1, 2], [1, 2, 3]), ([5, 6, 4, 3, 1, 2], [3, 2, 1]), ([6, 3, 2, 5, 4, 1], [1, 2, 3]), ([6, 4, 5, 2, 3, 1], [3, 2, 1])]),
    (6,6,4) : (14, [([3, 2, 1, 6, 5, 4], [4, 3, 2, 1]), ([4, 5, 6, 1, 2, 3], [1, 2, 3, 4])]),


}

permutation_list = genPermutationsBetween(1,5)
nu = combination_of_permutations([1,1],[[2,1],[1,2]])
nu2 = combination_of_permutations([1,1],[[1,2],[2,1]])
xi = combination_of_permutations([2,-2,-3],[[1,2,3],[3,2,1],[1,2]])

xi2 = combination_of_permutations.nested([[2,[1,2,3]],[-2,[3,2,1]],[-3,[1,2]]])

print(xi2.genFuzzyMatrix(3))

print(xi2.isConstant(3))

print(xi2.isVanishing(3))

"""
rho = combination_of_permutations.nested([[12,[3, 0, 2, 4, 1]], [-12,[1, 4, 2, 0, 3]], [5,[3, 2, 1, 0]], [10,[1, 3, 0, 2]], [5,[0, 1, 2, 3]]])
rho_matrices = rho.genFuzzyMatrixList(5)
for mat in rho_matrices:
    print(mat)
    print()
    
for something in rho.d4Symmetries(False):
    print(something)
    vec = something.make_vector(permutation_list)
    back_to = combination_of_permutations.vector(vec,permutation_list)
    print(back_to == something)"""
print()

In [3]:
ones = combination_of_permutations([1],[[1]])
nu = combination_of_permutations([1,1],[[1,2],[2,1]])
xi = combination_of_permutations([2,-2,-3],[[1,2,3],[3,2,1],[1,2]])
rho_star = combination_of_permutations([1,1,1,1,1/2,1/2],[[1,2,3],[3,2,1],[2,1,4,3],[3,4,1,2],[2,4,1,3],[3,1,4,2]])
Peter_cover = combination_of_permutations.nested([[1,[0,1,2,3]],[-4,[0,2,1,3]],[6,[0,2,3,1]],[6,[0,3,1,2]],[3,[1,0,2,3]],[-4,[0,2,1]]])
new_vecs = [(1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
            (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -4/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -6/5, 0, 0, 0, -6/5, -4/5, 0),
            (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -4/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -6/5, 0, -8/5, -8/5, 6/5, -4/5, 4/5),
            Peter_cover.make_vector(genPermutationsBetween(1,5)),
            Peter_cover.d4Symmetries()[1].make_vector(genPermutationsBetween(1,5)),
            (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -12/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -8/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -8/5, 0, 0, 0, 0, 0, 8/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -3/5, 0, 0, 0, -9/5, 0, 3, 0, 0, 0, 9/5, -4/5, -8/5),
            (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -8/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -12/5, 0, -8/5, 0, 0, 0, 0, 0, 8/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -9/5, 0, 0, 0, -3/5, 0, 3, 0, 0, 0, 9/5, -4/5, -8/5),
            (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -16/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -4/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -8/5, 0, 0, 0, 0, 0, 8/5, 0, 0, 0, 0, 0, -4/5, 0, 0, 0, 0, 0, -1/5, 0, 0, 0, -9/5, 0, 17/5, 0, 0, -8/5, 17/5, 8/5, -4),
            #(1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -24/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -24/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -24/5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -24/5, -24/5, 0, -24/5, 0, 0, 0, 0, 0, 48/5, 0, 0, 0, 0, 0, -24/5, 0, 0, 0, 0, 0, -24/5, 0, 0, -24/5, -24/5, 0, 96/5, 0, -24/5, -24/5, 96/5, 48/5, -24),
            ]
new_vec_CoPS = [combination_of_permutations.vector(vector(QQ,vec),genPermutationsBetween(1,5)) for vec in new_vecs]

L2_atomics = [
    [[1,[1,2]],[1,[2,1]]]
]

L3_atomics = [
    [[2,[1,2,3]],[-2,[3,2,1]],[-3,[1,2]]]
]

L4_atomics = [
    [
        [3,[3,1,4,2]],[3,[2,4,1,3]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [3,[3,4,1,2]],[3,[2,1,4,3]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [-3,[4,2,3,1]],[-3,[1,3,2,4]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [-3,[4,3,2,1]],[-3,[1,2,3,4]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
       [36,[5,2,3,4,1]],[-36,[1,2,3,4,5]],[15,[3,4,1,2]],[10,[1,2,3]] 
    ],

    [
       [36,[5,2,4,3,1]],[-36,[1,2,4,3,5]],[15,[3,4,1,2]],[10,[1,2,3]] 
    ]
]
   
L4_covers = [
    [
        [3,[3,1,4,2]],[3,[2,4,1,3]],[4,[1,2,3]],[3,[2,1]]
    ],
    [
        [3,[3,4,1,2]],[3,[2,1,4,3]],[4,[1,2,3]],[3,[2,1]]  
    ],
    [
        [3,[4,2,3,1]],[3,[1,3,2,4]],[-4,[1,2,3]],[3,[1,2]]
    ],
    [
        [-3,[4,2,3,1]],[-3,[1,3,2,4]],[4,[1,2,3]],[3,[2,1]]
    ],
    [
        [3,[4,3,2,1]],[3,[1,2,3,4]],[-4,[1,2,3]],[3,[1,2]]
    ],
    [
        [-3,[4,3,2,1]],[-3,[1,2,3,4]],[4,[1,2,3]],[3,[2,1]]
    ],
    [
        [3,[3,1,4,2]],[3,[2,4,1,3]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [3,[3,4,1,2]],[3,[2,1,4,3]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [-3,[4,2,3,1]],[-3,[1,3,2,4]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [-3,[4,3,2,1]],[-3,[1,2,3,4]],[2,[3,2,1]],[2,[1,2,3]]
    ],
    [
        [-36,[5,2,3,4,1]],[36,[1,2,3,4,5]],[15,[2,1,4,3]],[10,[3,2,1]]
    ],
    [
        [36,[5,2,3,4,1]],[-36,[1,2,3,4,5]],[15,[3,4,1,2]],[10,[1,2,3]] 
    ],
    [
        [-36,[5,2,4,3,1]],[36,[1,2,4,3,5]],[15,[2,1,4,3]],[10,[3,2,1]]
    ],
    [
       [36,[5,2,4,3,1]],[-36,[1,2,4,3,5]],[15,[3,4,1,2]],[10,[1,2,3]] 
    ]
]

L5_covers = [
    [[-3,[2, 0, 3, 1]], [-3,[1, 3, 0, 2]], [4,[1, 0, 2]], [4,[0, 2, 1]], [3,[1, 0]]],
    [[-3,[2, 3, 0, 1]], [-3,[1, 0, 3, 2]], [4,[1, 0, 2]], [4,[0, 2, 1]], [3,[1, 0]]],
    [[3,[3, 1, 2, 0]], [3,[0, 2, 1, 3]], [4,[1, 0, 2]], [4,[0, 2, 1]], [-3,[0, 1]]],
    [[3,[3, 1, 2, 0]], [3,[0, 2, 1, 3]], [4,[1, 0, 2]], [4,[0, 2, 1]], [3,[1, 0]]],
    [[3,[3, 2, 1, 0]], [3,[0, 1, 2, 3]], [4,[1, 0, 2]], [4,[0, 2, 1]], [-3,[0, 1]]],
    [[3,[3, 2, 1, 0]], [3,[0, 1, 2, 3]], [4,[1, 0, 2]], [4,[0, 2, 1]], [3,[1, 0]]],
    [[3,[3, 1, 2, 0]], [3,[0, 2, 1, 3]], [2,[1, 0, 2]], [2,[0, 2, 1]], [-2,[0, 1, 2]]],
    [[3,[3, 2, 1, 0]], [3,[0, 1, 2, 3]], [2,[1, 0, 2]], [2,[0, 2, 1]], [-2,[0, 1, 2]]],
    [[-3,[1, 0, 3, 2]], [-4,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [4,[2, 1, 0]], [6,[0, 1]]],
    [[3,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [-4,[2, 1, 0]], [6,[1, 0]]],
    [[3,[2, 3, 0, 1]], [-4,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [8,[0, 1, 2]], [3,[1, 0]]],
    [[3,[2, 3, 0, 1]], [-4,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [8,[2, 1, 0]], [9,[0, 1]]],
    [[-3,[2, 3, 0, 1]], [4,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [-8,[2, 1, 0]], [9,[1, 0]]],
    [[12,[3, 1, 2, 0]], [-9,[1, 0, 3, 2]], [-15,[0, 1, 2, 3]], [-4,[0, 1, 2]], [12,[0, 1]]],
    [[-12,[3, 1, 2, 0]], [9,[1, 0, 3, 2]], [15,[0, 1, 2, 3]], [4,[0, 1, 2]], [12,[1, 0]]],
    [[12,[3, 1, 2, 0]], [-9,[1, 0, 3, 2]], [-15,[0, 1, 2, 3]], [-4,[2, 1, 0]], [6,[0, 1]]],
    [[-12,[3, 1, 2, 0]], [9,[1, 0, 3, 2]], [15,[0, 1, 2, 3]], [4,[2, 1, 0]], [6,[1, 0]]],
    [[12,[3, 1, 2, 0]], [9,[2, 3, 0, 1]], [-15,[0, 1, 2, 3]], [8,[0, 1, 2]], [3,[0, 1]]],
    [[12,[3, 1, 2, 0]], [9,[2, 3, 0, 1]], [-15,[0, 1, 2, 3]], [8,[0, 1, 2]], [-3,[1, 0]]],
    [[12,[3, 1, 2, 0]], [9,[2, 3, 0, 1]], [-15,[0, 1, 2, 3]], [8,[2, 1, 0]], [15,[0, 1]]],
    [[-12,[3, 1, 2, 0]], [-9,[2, 3, 0, 1]], [15,[0, 1, 2, 3]], [-8,[2, 1, 0]], [15,[1, 0]]],
    [[3,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [4,[2, 0, 1]], [4,[1, 2, 0]]],
    [[3,[2, 3, 0, 1]], [-4,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [2,[2, 1, 0]], [6,[0, 1, 2]]],
    [[-12,[3, 1, 2, 0]], [9,[1, 0, 3, 2]], [15,[0, 1, 2, 3]], [8,[2, 1, 0]], [-4,[0, 1, 2]]],
    [[12,[3, 1, 2, 0]], [9,[2, 3, 0, 1]], [-15,[0, 1, 2, 3]], [-2,[2, 1, 0]], [10,[0, 1, 2]]],
    [[3,[2, 3, 0, 1]], [6,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [3,[1, 0]]],
    [[3,[3, 1, 2, 0]], [-3,[1, 0, 3, 2]], [-1,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [3,[0, 1]]],
    [[-3,[3, 1, 2, 0]], [3,[1, 0, 3, 2]],  [1,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [3,[1, 0]]],
    [[6,[3, 1, 2, 0]], [3,[2, 3, 0, 1]], [2,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [3,[0, 1]]],
    [[-6,[3, 1, 2, 0]], [-3,[2, 3, 0, 1]], [-2,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [3,[1, 0]]],
    [[4,[3, 1, 2, 0]],  [1,[2, 3, 0, 1]], [-2,[1, 0, 3, 2]], [-5,[0, 1, 2, 3]], [3,[0, 1]]],
    [[-4,[3, 1, 2, 0]], [-1,[2, 3, 0, 1]], [2,[1, 0, 3, 2]], [5,[0, 1, 2, 3]], [3,[1, 0]]],
    [[3,[3, 2, 1, 0]], [-3,[1, 0, 3, 2]], [-4,[0, 2, 1, 3]], [-2,[0, 1, 2, 3]], [3,[0, 1]]],
    [[-3,[3, 2, 1, 0]], [3,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [2,[0, 1, 2, 3]], [3,[1, 0]]],
    [[6,[3, 2, 1, 0]], [3,[2, 3, 0, 1]], [-4,[0, 2, 1, 3]],  [1,[0, 1, 2, 3]], [3,[0, 1]]],
    [[-6,[3, 2, 1, 0]], [-3,[2, 3, 0, 1]], [4,[0, 2, 1, 3]], [-1,[0, 1, 2, 3]], [3,[1, 0]]],
    [[6,[2, 3, 0, 1]], [9,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [4,[2, 1, 0]]],
    [[-6,[3, 1, 2, 0]], [3,[1, 0, 3, 2]], [-2,[0, 2, 1, 3]], [5,[0, 1, 2, 3]], [4,[2, 1, 0]]],
    [[3,[3, 1, 2, 0]], [3,[2, 3, 0, 1]], [-1,[0, 2, 1, 3]], [-5,[0, 1, 2, 3]], [4,[0, 1, 2]]],
    [[-4,[3, 1, 2, 0]], [2,[2, 3, 0, 1]], [5,[1, 0, 2, 3]], [5,[0, 1, 3, 2]], [4,[2, 1, 0]]],
    [[4,[3, 1, 2, 0]], [4,[2, 3, 0, 1]],  [1,[1, 0, 3, 2]], [-5,[0, 1, 2, 3]], [4,[0, 1, 2]]],
    [[-4,[3, 1, 2, 0]], [2,[2, 3, 0, 1]], [5,[1, 0, 3, 2]], [5,[0, 1, 2, 3]], [4,[2, 1, 0]]],
    [[3,[3, 2, 0, 1]], [3,[2, 3, 1, 0]], [-4,[0, 2, 1, 3]], [-2,[0, 1, 2, 3]], [4,[0, 1, 2]]],
    [[-6,[3, 2, 1, 0]], [3,[1, 0, 3, 2]], [4,[0, 2, 1, 3]], [-1,[0, 1, 2, 3]], [4,[2, 1, 0]]],
    [[3,[3, 2, 1, 0]], [3,[2, 3, 0, 1]], [-4,[0, 2, 1, 3]], [-2,[0, 1, 2, 3]], [4,[0, 1, 2]]],
    [[36,[3, 0, 2, 4, 1]], [-36,[1, 4, 2, 0, 3]], [30,[1, 3, 0, 2]], [20,[0, 1, 2]], [15,[1, 0]]],
    [[36,[4, 1, 2, 3, 0]], [-36,[0, 1, 2, 3, 4]], [-15,[1, 0, 3, 2]], [-10,[0, 1, 2]], [15,[0, 1]]],
    [[-36,[4, 1, 2, 3, 0]], [36,[0, 1, 2, 3, 4]], [15,[1, 0, 3, 2]], [10,[0, 1, 2]], [15,[1, 0]]],
    [[36,[4, 1, 2, 3, 0]], [-36,[0, 1, 2, 3, 4]], [15,[2, 3, 0, 1]], [10,[2, 1, 0]], [15,[0, 1]]],
    [[-36,[4, 1, 2, 3, 0]], [36,[0, 1, 2, 3, 4]], [-15,[2, 3, 0, 1]], [-10,[2, 1, 0]], [15,[1, 0]]],
    [[36,[4, 1, 3, 2, 0]], [-36,[0, 1, 3, 2, 4]], [-15,[1, 0, 3, 2]], [-10,[0, 1, 2]], [15,[0, 1]]],
    [[-36,[4, 1, 3, 2, 0]], [36,[0, 1, 3, 2, 4]], [15,[1, 0, 3, 2]], [10,[0, 1, 2]], [15,[1, 0]]],
    [[36,[4, 1, 3, 2, 0]], [-36,[0, 1, 3, 2, 4]], [15,[2, 3, 0, 1]], [10,[2, 1, 0]], [15,[0, 1]]],
    [[-36,[4, 1, 3, 2, 0]], [36,[0, 1, 3, 2, 4]], [-15,[2, 3, 0, 1]], [-10,[2, 1, 0]], [15,[1, 0]]],
    [[12,[4, 3, 2, 1, 0]], [-12,[0, 1, 2, 3, 4]], [15,[0, 1, 2, 3]], [-10,[0, 1, 2]], [5,[0, 1]]],
    [[-12,[4, 3, 2, 1, 0]], [12,[0, 1, 2, 3, 4]], [-15,[0, 1, 2, 3]], [10,[0, 1, 2]], [5,[1, 0]]],
    [[-12,[4, 3, 2, 1, 0]], [12,[0, 1, 2, 3, 4]], [-15,[0, 1, 2, 3]], [10,[2, 1, 0]], [10,[0, 1]]],
    [[12,[4, 3, 2, 1, 0]], [-12,[0, 1, 2, 3, 4]], [15,[0, 1, 2, 3]], [-10,[2, 1, 0]], [10,[1, 0]]],
    [[18,[3, 0, 2, 4, 1]], [-18,[1, 4, 2, 0, 3]], [15,[1, 3, 0, 2]], [5,[2, 1, 0]], [5,[0, 1, 2]]],
    [[36,[4, 1, 2, 3, 0]], [-36,[0, 1, 2, 3, 4]], [-15,[1, 0, 3, 2]], [10,[1, 0, 2]], [10,[0, 2, 1]]],
    [[-36,[4, 1, 2, 3, 0]], [36,[0, 1, 2, 3, 4]], [-15,[2, 3, 0, 1]], [10,[2, 0, 1]], [10,[1, 2, 0]]],
    [[36,[4, 1, 3, 2, 0]], [-36,[0, 1, 3, 2, 4]], [-15,[1, 0, 3, 2]], [10,[1, 0, 2]], [10,[0, 2, 1]]],
    [[-36,[4, 1, 3, 2, 0]], [36,[0, 1, 3, 2, 4]], [-15,[2, 3, 0, 1]], [10,[2, 0, 1]], [10,[1, 2, 0]]],
    [[-36,[4, 3, 2, 1, 0]], [36,[0, 1, 2, 3, 4]], [-45,[0, 1, 2, 3]], [10,[2, 1, 0]], [20,[0, 1, 2]]],
    [[-24,[3, 4, 2, 0, 1]], [24,[1, 0, 2, 4, 3]], [15,[2, 1, 0, 3]], [15,[0, 3, 2, 1]], [5,[1, 0]]],
    [[-24,[3, 4, 2, 0, 1]], [24,[1, 0, 2, 4, 3]], [15,[2, 3, 0, 1]], [15,[0, 1, 2, 3]], [5,[1, 0]]],
    [[24,[4, 1, 2, 3, 0]], [-24,[0, 1, 2, 3, 4]], [5,[2, 3, 0, 1]], [-5,[1, 0, 3, 2]], [5,[0, 1]]],
    [[-24,[4, 1, 2, 3, 0]], [24,[0, 1, 2, 3, 4]], [-5,[2, 3, 0, 1]], [5,[1, 0, 3, 2]], [5,[1, 0]]],
    [[24,[4, 1, 3, 2, 0]], [-24,[0, 1, 3, 2, 4]], [5,[2, 3, 0, 1]], [-5,[1, 0, 3, 2]], [5,[0, 1]]],
    [[-24,[4, 1, 3, 2, 0]], [24,[0, 1, 3, 2, 4]], [-5,[2, 3, 0, 1]], [5,[1, 0, 3, 2]], [5,[1, 0]]],
    [[-24,[4, 3, 2, 1, 0]], [24,[0, 1, 2, 3, 4]], [15,[3, 2, 1, 0]], [-15,[0, 1, 2, 3]], [5,[0, 1]]],
    [[-36,[4, 3, 2, 1, 0]], [36,[0, 1, 2, 3, 4]], [15,[3, 2, 1, 0]], [-30,[0, 1, 2, 3]], [10,[0, 1, 2]]],
    [[12,[3, 0, 2, 4, 1]], [-12,[1, 4, 2, 0, 3]], [5,[3, 1, 2, 0]], [10,[1, 3, 0, 2]], [5,[0, 2, 1, 3]]],
    [[12,[3, 0, 2, 4, 1]], [-12,[1, 4, 2, 0, 3]], [5,[3, 2, 1, 0]], [10,[1, 3, 0, 2]], [5,[0, 1, 2, 3]]]
            ]

upto4_term_atomics_non_square = [
[[1, [2, 1]], [1, [1, 2]]],
[[-2, [3, 2, 1]], [2, [1, 2, 3]], [-3, [1, 2]]],
[[1, [3, 2, 1]], [1, [2, 1, 3]], [1, [1, 3, 2]]],
[[1, [3, 1, 2]], [1, [2, 3, 1]], [1, [1, 2, 3]]],
[[3, [3, 1, 4, 2]], [3, [2, 4, 1, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
[[3, [3, 4, 1, 2]], [3, [2, 1, 4, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
[[-3, [4, 2, 3, 1]], [-3, [1, 3, 2, 4]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
[[-3, [4, 3, 2, 1]], [-3, [1, 2, 3, 4]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
[[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
[[-36, [5, 4, 3, 2, 1]], [36, [1, 4, 3, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[36, [5, 2, 4, 3, 1]], [-36, [1, 2, 4, 3, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
[[-36, [5, 4, 2, 3, 1]], [36, [1, 4, 2, 3, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[-36, [5, 3, 4, 2, 1]], [36, [1, 3, 4, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[36, [5, 3, 2, 4, 1]], [-36, [1, 3, 2, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
[[1, [4, 2, 1, 3]], [1, [3, 4, 2, 1]], [1, [2, 1, 3, 4]], [1, [1, 3, 4, 2]]],
[[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 3, 4]], [1, [1, 2, 4, 3]]],
[[1, [4, 3, 1, 2]], [1, [3, 4, 2, 1]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
[[1, [4, 3, 2, 1]], [1, [3, 2, 1, 4]], [1, [2, 1, 4, 3]], [1, [1, 4, 3, 2]]],
[[1, [4, 3, 1, 2]], [1, [3, 2, 4, 1]], [1, [2, 1, 3, 4]], [1, [1, 4, 2, 3]]],
[[1, [4, 3, 1, 2]], [1, [3, 1, 2, 4]], [1, [2, 4, 3, 1]], [1, [1, 2, 4, 3]]],
[[1, [4, 2, 3, 1]], [1, [3, 1, 2, 4]], [1, [2, 4, 1, 3]], [1, [1, 3, 4, 2]]],
[[1, [4, 2, 3, 1]], [1, [3, 1, 4, 2]], [1, [2, 3, 1, 4]], [1, [1, 4, 2, 3]]],
[[1, [4, 2, 1, 3]], [1, [3, 1, 2, 4]], [1, [2, 3, 4, 1]], [1, [1, 4, 3, 2]]],
[[1, [4, 1, 3, 2]], [1, [3, 4, 2, 1]], [1, [2, 3, 1, 4]], [1, [1, 2, 4, 3]]],
[[1, [4, 1, 2, 3]], [1, [3, 4, 1, 2]], [1, [2, 3, 4, 1]], [1, [1, 2, 3, 4]]],
[[1, [4, 1, 2, 3]], [1, [3, 2, 4, 1]], [1, [2, 3, 1, 4]], [1, [1, 4, 3, 2]]],
]

upto4_term_atomics_square = [
[[1, [2, 1]], [1, [1, 2]]],
[[1, [3, 2, 1]], [1, [2, 1, 3]], [1, [1, 3, 2]]],
[[1, [3, 1, 2]], [1, [2, 3, 1]], [1, [1, 2, 3]]],
[[-2, [3, 2, 1]], [2, [1, 2, 3]], [-3, [1, 2]]],
[[1, [4, 2, 3, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 3, 2, 4]]],
[[1, [4, 2, 1, 3]], [1, [3, 4, 2, 1]], [1, [2, 1, 3, 4]], [1, [1, 3, 4, 2]]],
[[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
[[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 3, 4]], [1, [1, 2, 4, 3]]],
[[1, [4, 3, 1, 2]], [1, [3, 4, 2, 1]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
[[1, [4, 3, 2, 1]], [1, [3, 2, 1, 4]], [1, [2, 1, 4, 3]], [1, [1, 4, 3, 2]]],
[[1, [4, 3, 1, 2]], [1, [3, 2, 4, 1]], [1, [2, 1, 3, 4]], [1, [1, 4, 2, 3]]],
[[1, [4, 3, 2, 1]], [1, [3, 1, 4, 2]], [1, [2, 4, 1, 3]], [1, [1, 2, 3, 4]]],
[[1, [4, 3, 1, 2]], [1, [3, 1, 2, 4]], [1, [2, 4, 3, 1]], [1, [1, 2, 4, 3]]],
[[1, [4, 2, 3, 1]], [1, [3, 1, 2, 4]], [1, [2, 4, 1, 3]], [1, [1, 3, 4, 2]]],
[[1, [4, 2, 3, 1]], [1, [3, 1, 4, 2]], [1, [2, 3, 1, 4]], [1, [1, 4, 2, 3]]],
[[1, [4, 2, 1, 3]], [1, [3, 1, 2, 4]], [1, [2, 3, 4, 1]], [1, [1, 4, 3, 2]]],
[[1, [4, 1, 3, 2]], [1, [3, 4, 2, 1]], [1, [2, 3, 1, 4]], [1, [1, 2, 4, 3]]],
[[1, [4, 1, 2, 3]], [1, [3, 4, 1, 2]], [1, [2, 3, 4, 1]], [1, [1, 2, 3, 4]]],
[[1, [4, 1, 2, 3]], [1, [3, 2, 4, 1]], [1, [2, 3, 1, 4]], [1, [1, 4, 3, 2]]],
[[3, [3, 1, 4, 2]], [3, [2, 4, 1, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
[[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
[[-36, [5, 4, 3, 2, 1]], [36, [1, 4, 3, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[36, [5, 2, 4, 3, 1]], [-36, [1, 2, 4, 3, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
[[-36, [5, 4, 2, 3, 1]], [36, [1, 4, 2, 3, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[-36, [5, 3, 4, 2, 1]], [36, [1, 3, 4, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
[[36, [5, 3, 2, 4, 1]], [-36, [1, 3, 2, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
]

mixed_len_prev_known = {
    2 : L2_atomics,
    3: L3_atomics,
    4: L4_atomics,
    5: L5_covers
}

upto5_term_atomics_non_square = [
    [[1, [2, 1]], [1, [1, 2]]],
    [[-2, [3, 2, 1]], [2, [1, 2, 3]], [-3, [1, 2]]],
    [[1, [3, 2, 1]], [1, [2, 1, 3]], [1, [1, 3, 2]]],
    [[1, [3, 1, 2]], [1, [2, 3, 1]], [1, [1, 2, 3]]],
    [[3, [3, 1, 4, 2]], [3, [2, 4, 1, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[3, [3, 4, 1, 2]], [3, [2, 1, 4, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[-3, [4, 2, 3, 1]], [-3, [1, 3, 2, 4]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[-3, [4, 3, 2, 1]], [-3, [1, 2, 3, 4]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 3, 2, 1]], [36, [1, 4, 3, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 2, 4, 3, 1]], [-36, [1, 2, 4, 3, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 2, 3, 1]], [36, [1, 4, 2, 3, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[-36, [5, 3, 4, 2, 1]], [36, [1, 3, 4, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 3, 2, 4, 1]], [-36, [1, 3, 2, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[1, [4, 2, 1, 3]], [1, [3, 4, 2, 1]], [1, [2, 1, 3, 4]], [1, [1, 3, 4, 2]]],
    [[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 3, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 3, 1, 2]], [1, [3, 4, 2, 1]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 2, 1]], [1, [3, 2, 1, 4]], [1, [2, 1, 4, 3]], [1, [1, 4, 3, 2]]],
    [[1, [4, 3, 1, 2]], [1, [3, 2, 4, 1]], [1, [2, 1, 3, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 3, 1, 2]], [1, [3, 1, 2, 4]], [1, [2, 4, 3, 1]], [1, [1, 2, 4, 3]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 2, 4]], [1, [2, 4, 1, 3]], [1, [1, 3, 4, 2]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 4, 2]], [1, [2, 3, 1, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 2, 1, 3]], [1, [3, 1, 2, 4]], [1, [2, 3, 4, 1]], [1, [1, 4, 3, 2]]],
    [[1, [4, 1, 3, 2]], [1, [3, 4, 2, 1]], [1, [2, 3, 1, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 1, 2, 3]], [1, [3, 4, 1, 2]], [1, [2, 3, 4, 1]], [1, [1, 2, 3, 4]]],
    [[1, [4, 1, 2, 3]], [1, [3, 2, 4, 1]], [1, [2, 3, 1, 4]], [1, [1, 4, 3, 2]]],
    [[-3, [2, 1, 4, 3]], [-4, [1, 3, 2, 4]], [-5, [1, 2, 3, 4]], [4, [3, 2, 1]], [6, [1, 2]]],
    [[36, [4, 1, 3, 5, 2]], [-36, [2, 5, 3, 1, 4]], [30, [2, 4, 1, 3]], [20, [1, 2, 3]], [15, [2, 1]]],
    [[12, [5, 4, 3, 2, 1]], [-12, [1, 2, 3, 4, 5]], [15, [1, 2, 3, 4]], [-10, [1, 2, 3]], [5, [1, 2]]],
    [[-24, [4, 5, 3, 1, 2]], [24, [2, 1, 3, 5, 4]], [15, [3, 2, 1, 4]], [15, [1, 4, 3, 2]], [5, [2, 1]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 5, 2, 1]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 2, 1, 5]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 2, 5, 1]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 2, 1, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 1, 4, 3]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 1, 3, 4]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 5, 2, 1]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 2, 4, 1, 5]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 2, 4, 5, 1]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 1, 3, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 1, 2, 3]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 5, 1, 3]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 2, 5, 3, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 5, 1, 3]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 3, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 2, 3, 5, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 1, 5, 3]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 4, 2, 1]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 2, 1, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 2, 1, 5, 3]], [1, [3, 5, 2, 4, 1]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 1, 2, 3]], [1, [4, 2, 5, 3, 1]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 5, 4, 3, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 5, 4, 2, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 5, 4, 2, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 5, 4, 3, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 2, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 3, 2, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 3, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 1, 5, 2]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 2, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 1, 2, 5]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 4, 2, 5, 3]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 5, 3, 2]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 4, 5, 2, 3]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 4, 3, 5, 2]]],
    [[1, [5, 3, 2, 1, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 5, 3, 2]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 1, 4, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 1, 2, 4]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 1, 5, 4, 2]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 1, 5, 2, 4]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 2, 4, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 5, 2, 4]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 3, 2, 4, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 4, 2, 5]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 3, 2, 1, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 1, 4, 5, 2]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 5, 1, 3]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 5, 3, 1]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 3, 1, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 4, 5]], [1, [2, 4, 3, 5, 1]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 1, 5, 4, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 1, 5, 2, 4]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 3, 5, 1, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 5, 4]], [1, [2, 3, 5, 4, 1]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 3, 4, 1, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 4, 5]], [1, [2, 3, 4, 5, 1]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 1, 4, 2, 5]], [1, [2, 5, 1, 4, 3]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 1, 4, 5, 2]], [1, [2, 5, 1, 3, 4]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 1, 5, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 5, 4, 1, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 1, 2, 5]], [1, [3, 1, 2, 5, 4]], [1, [2, 5, 4, 3, 1]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 1, 2, 5]], [1, [3, 1, 5, 4, 2]], [1, [2, 5, 3, 1, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 1, 5, 2]], [1, [3, 1, 5, 2, 4]], [1, [2, 5, 3, 4, 1]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 5, 3, 2]], [1, [3, 5, 2, 1, 4]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 1, 5, 2, 3]], [1, [3, 5, 2, 4, 1]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 2, 5, 3]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 2, 3, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 1, 3, 2, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 1, 3, 5, 2]], [1, [3, 5, 4, 2, 1]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 1, 3, 2, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 1, 3, 4, 2]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 5, 2, 1]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 1, 2, 4, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 1, 2, 3, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 1, 4, 3, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 2, 1, 5]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 1, 4, 2, 3]], [1, [4, 5, 3, 1, 2]], [1, [3, 4, 2, 5, 1]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]]
]

upto5_term_atomics_square = [
    [[1, [2, 1]], [1, [1, 2]]],
    [[1, [3, 2, 1]], [1, [2, 1, 3]], [1, [1, 3, 2]]],
    [[1, [3, 1, 2]], [1, [2, 3, 1]], [1, [1, 2, 3]]],
    [[-2, [3, 2, 1]], [2, [1, 2, 3]], [-3, [1, 2]]],
    [[1, [4, 2, 3, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 3, 2, 4]]],
    [[1, [4, 2, 1, 3]], [1, [3, 4, 2, 1]], [1, [2, 1, 3, 4]], [1, [1, 3, 4, 2]]],
    [[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 3, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 3, 1, 2]], [1, [3, 4, 2, 1]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 2, 1]], [1, [3, 2, 1, 4]], [1, [2, 1, 4, 3]], [1, [1, 4, 3, 2]]],
    [[1, [4, 3, 1, 2]], [1, [3, 2, 4, 1]], [1, [2, 1, 3, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 3, 2, 1]], [1, [3, 1, 4, 2]], [1, [2, 4, 1, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 1, 2]], [1, [3, 1, 2, 4]], [1, [2, 4, 3, 1]], [1, [1, 2, 4, 3]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 2, 4]], [1, [2, 4, 1, 3]], [1, [1, 3, 4, 2]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 4, 2]], [1, [2, 3, 1, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 2, 1, 3]], [1, [3, 1, 2, 4]], [1, [2, 3, 4, 1]], [1, [1, 4, 3, 2]]],
    [[1, [4, 1, 3, 2]], [1, [3, 4, 2, 1]], [1, [2, 3, 1, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 1, 2, 3]], [1, [3, 4, 1, 2]], [1, [2, 3, 4, 1]], [1, [1, 2, 3, 4]]],
    [[1, [4, 1, 2, 3]], [1, [3, 2, 4, 1]], [1, [2, 3, 1, 4]], [1, [1, 4, 3, 2]]],
    [[3, [3, 1, 4, 2]], [3, [2, 4, 1, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 3, 2, 1]], [36, [1, 4, 3, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 2, 4, 3, 1]], [-36, [1, 2, 4, 3, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 2, 3, 1]], [36, [1, 4, 2, 3, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[-36, [5, 3, 4, 2, 1]], [36, [1, 3, 4, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 3, 2, 4, 1]], [-36, [1, 3, 2, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 1, 5, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 1, 2, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 5, 2, 1]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 5, 1, 2, 3]], [1, [3, 4, 2, 1, 5]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 2, 4, 1, 3]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 2, 5, 1]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 4, 2, 1, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 2, 1, 4, 3]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 2, 1, 3, 4]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 5, 2, 1]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 2, 4, 1, 5]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 2, 4, 5, 1]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 1, 3, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 1, 2, 3]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 5, 1, 3]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 2, 5, 3, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 5, 1, 3]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 3, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 2, 3, 5, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 2, 1, 5, 3]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 3, 4, 5]], [1, [1, 3, 5, 2, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 4, 2, 1]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 5, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 2, 5, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 2, 1, 3, 5]], [1, [3, 5, 2, 1, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 3, 4, 5, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 2, 1, 5, 3]], [1, [3, 5, 2, 4, 1]], [1, [2, 1, 5, 3, 4]], [1, [1, 3, 4, 2, 5]]],
    [[1, [5, 4, 1, 2, 3]], [1, [4, 2, 5, 3, 1]], [1, [3, 5, 4, 1, 2]], [1, [2, 1, 3, 5, 4]], [1, [1, 3, 2, 4, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 5, 4, 3, 2]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 5, 3, 4]], [1, [1, 5, 4, 2, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 5, 4, 2, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 5, 4, 3, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 2, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 3, 2, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 3, 4, 2]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 1, 5, 2]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 5, 2, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 1, 2, 5]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 5, 3]], [1, [1, 5, 2, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 5, 3, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 3, 4, 5]], [1, [1, 4, 2, 5, 3]]],
    [[1, [5, 2, 3, 4, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 5, 3, 2]]],
    [[1, [5, 2, 3, 1, 4]], [1, [4, 3, 2, 5, 1]], [1, [3, 5, 1, 4, 2]], [1, [2, 1, 4, 3, 5]], [1, [1, 4, 5, 2, 3]]],
    [[1, [5, 2, 4, 3, 1]], [1, [4, 3, 2, 1, 5]], [1, [3, 5, 1, 2, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 4, 3, 5, 2]]],
    [[1, [5, 3, 2, 1, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 4, 5]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 5, 3, 2]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 1, 5, 4]], [1, [2, 1, 5, 4, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 1, 4, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 2, 5, 1, 4]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 1, 2, 4]], [1, [4, 5, 3, 1, 2]], [1, [3, 2, 5, 4, 1]], [1, [2, 1, 4, 5, 3]], [1, [1, 4, 2, 3, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 1, 5, 4, 2]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 1, 5, 2, 4]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 2, 4, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 5, 2, 4]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 3, 2, 4, 1]], [1, [4, 5, 3, 1, 2]], [1, [3, 1, 4, 2, 5]], [1, [2, 4, 1, 5, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 3, 2, 1, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 1, 4, 5, 2]], [1, [2, 4, 1, 3, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 5, 1, 3]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 5, 3, 1]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 3, 4, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 4, 3, 1, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 3, 4, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 4, 5]], [1, [2, 4, 3, 5, 1]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 2, 1, 3]], [1, [3, 1, 5, 4, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 2, 3, 1]], [1, [3, 1, 5, 2, 4]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 3, 5, 1, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 5, 4]], [1, [2, 3, 5, 4, 1]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 5, 1, 3, 2]], [1, [3, 1, 2, 5, 4]], [1, [2, 3, 4, 1, 5]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 5, 1, 2, 3]], [1, [3, 1, 2, 4, 5]], [1, [2, 3, 4, 5, 1]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 5, 1, 2]], [1, [3, 1, 4, 2, 5]], [1, [2, 5, 1, 4, 3]], [1, [1, 2, 3, 5, 4]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 5, 2, 1]], [1, [3, 1, 4, 5, 2]], [1, [2, 5, 1, 3, 4]], [1, [1, 2, 3, 4, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 3, 1, 5, 2]], [1, [3, 1, 2, 4, 5]], [1, [2, 5, 4, 1, 3]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 3, 1, 2, 5]], [1, [3, 1, 2, 5, 4]], [1, [2, 5, 4, 3, 1]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 3, 1, 2, 5]], [1, [3, 1, 5, 4, 2]], [1, [2, 5, 3, 1, 4]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 3, 1, 5, 2]], [1, [3, 1, 5, 2, 4]], [1, [2, 5, 3, 4, 1]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 5, 3, 2]], [1, [3, 5, 2, 1, 4]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 4, 3, 1, 2]], [1, [4, 1, 5, 2, 3]], [1, [3, 5, 2, 4, 1]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 2, 5, 3]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 4, 3, 2, 1]], [1, [4, 1, 2, 3, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 3, 1]], [1, [4, 1, 3, 2, 5]], [1, [3, 5, 4, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 4, 2, 1, 3]], [1, [4, 1, 3, 5, 2]], [1, [3, 5, 4, 2, 1]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]],
    [[1, [5, 1, 3, 2, 4]], [1, [4, 5, 2, 3, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 1, 3, 4, 2]], [1, [4, 5, 2, 1, 3]], [1, [3, 4, 5, 2, 1]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 1, 2, 4, 3]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 4, 3, 5]]],
    [[1, [5, 1, 2, 3, 4]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 5, 1, 2]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 4, 5, 3]]],
    [[1, [5, 1, 4, 3, 2]], [1, [4, 5, 3, 2, 1]], [1, [3, 4, 2, 1, 5]], [1, [2, 3, 1, 5, 4]], [1, [1, 2, 5, 4, 3]]],
    [[1, [5, 1, 4, 2, 3]], [1, [4, 5, 3, 1, 2]], [1, [3, 4, 2, 5, 1]], [1, [2, 3, 1, 4, 5]], [1, [1, 2, 5, 3, 4]]],
    [[-3, [2, 1, 4, 3]], [-4, [1, 3, 2, 4]], [-5, [1, 2, 3, 4]], [4, [3, 2, 1]], [6, [1, 2]]],
    [[36, [4, 1, 3, 5, 2]], [-36, [2, 5, 3, 1, 4]], [30, [2, 4, 1, 3]], [20, [1, 2, 3]], [15, [2, 1]]],
    [[12, [5, 4, 3, 2, 1]], [-12, [1, 2, 3, 4, 5]], [15, [1, 2, 3, 4]], [-10, [1, 2, 3]], [5, [1, 2]]],
    [[-24, [4, 5, 3, 1, 2]], [24, [2, 1, 3, 5, 4]], [15, [3, 2, 1, 4]], [15, [1, 4, 3, 2]], [5, [2, 1]]]
]

len(upto5_term_atomics_square)

129

In [2]:
n_at_most_4_square = [[[1, [0, 1]], [1, [1, 0]]],
[[1, [0, 2, 1]], [1, [1, 0, 2]], [1, [2, 1, 0]]],
[[1, [0, 1, 2]], [1, [1, 2, 0]], [1, [2, 0, 1]]],
[[-3, [0, 1]], [2, [0, 1, 2]], [-2, [2, 1, 0]]],
[[1, [0, 2, 1, 3]], [1, [1, 0, 3, 2]], [1, [2, 3, 0, 1]], [1, [3, 1, 2, 0]]],
[[1, [0, 2, 3, 1]], [1, [1, 0, 2, 3]], [1, [2, 3, 1, 0]], [1, [3, 1, 0, 2]]],
[[1, [0, 1, 2, 3]], [1, [1, 0, 3, 2]], [1, [2, 3, 0, 1]], [1, [3, 2, 1, 0]]],
[[1, [0, 1, 3, 2]], [1, [1, 0, 2, 3]], [1, [2, 3, 0, 1]], [1, [3, 2, 1, 0]]],
[[1, [0, 1, 2, 3]], [1, [1, 0, 3, 2]], [1, [2, 3, 1, 0]], [1, [3, 2, 0, 1]]],
[[1, [0, 3, 2, 1]], [1, [1, 0, 3, 2]], [1, [2, 1, 0, 3]], [1, [3, 2, 1, 0]]],
[[1, [0, 3, 1, 2]], [1, [1, 0, 2, 3]], [1, [2, 1, 3, 0]], [1, [3, 2, 0, 1]]],
[[1, [0, 1, 2, 3]], [1, [1, 3, 0, 2]], [1, [2, 0, 3, 1]], [1, [3, 2, 1, 0]]],
[[1, [0, 1, 3, 2]], [1, [1, 3, 2, 0]], [1, [2, 0, 1, 3]], [1, [3, 2, 0, 1]]],
[[1, [0, 2, 3, 1]], [1, [1, 3, 0, 2]], [1, [2, 0, 1, 3]], [1, [3, 1, 2, 0]]],
[[1, [0, 3, 1, 2]], [1, [1, 2, 0, 3]], [1, [2, 0, 3, 1]], [1, [3, 1, 2, 0]]],
[[1, [0, 3, 2, 1]], [1, [1, 2, 3, 0]], [1, [2, 0, 1, 3]], [1, [3, 1, 0, 2]]],
[[1, [0, 1, 3, 2]], [1, [1, 2, 0, 3]], [1, [2, 3, 1, 0]], [1, [3, 0, 2, 1]]],
[[1, [0, 1, 2, 3]], [1, [1, 2, 3, 0]], [1, [2, 3, 0, 1]], [1, [3, 0, 1, 2]]],
[[1, [0, 3, 2, 1]], [1, [1, 2, 0, 3]], [1, [2, 1, 3, 0]], [1, [3, 0, 1, 2]]],
[[2, [0, 1, 2]], [2, [2, 1, 0]], [3, [1, 3, 0, 2]], [3, [2, 0, 3, 1]]],
[[6, [0, 1]], [4, [2, 1, 0]], [-5, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [-3, [1, 0, 3, 2]]],
[[-4, [0, 2, 1]], [1, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [6, [0, 2, 3, 1]], [6, [0, 3, 1, 2]], [3, [1, 0, 2, 3]]],
[[-4, [2, 0, 1]], [3, [2, 3, 1, 0]], [6, [3, 0, 2, 1]], [6, [3, 1, 0, 2]], [-4, [3, 1, 2, 0]], [1, [3, 2, 1, 0]]]]

dim_20_n_at_most_4_square = n_at_most_4_square[:-3]
dim_21_n_at_most_4_square = n_at_most_4_square[:-2]


n_at_most_4_non_square = [[[1, [0, 1]], [1, [1, 0]]],
[[-3, [0, 1]], [2, [0, 1, 2]], [-2, [2, 1, 0]]],
[[1, [0, 2, 1]], [1, [1, 0, 2]], [1, [2, 1, 0]]],
[[1, [0, 1, 2]], [1, [1, 2, 0]], [1, [2, 0, 1]]],
[[2, [0, 1, 2]], [2, [2, 1, 0]], [3, [1, 3, 0, 2]], [3, [2, 0, 3, 1]]],
[[2, [0, 1, 2]], [2, [2, 1, 0]], [3, [1, 0, 3, 2]], [3, [2, 3, 0, 1]]],
[[2, [0, 1, 2]], [2, [2, 1, 0]], [-3, [0, 2, 1, 3]], [-3, [3, 1, 2, 0]]],
[[2, [0, 1, 2]], [2, [2, 1, 0]], [-3, [0, 1, 2, 3]], [-3, [3, 2, 1, 0]]],
[[1, [0, 2, 3, 1]], [1, [1, 0, 2, 3]], [1, [2, 3, 1, 0]], [1, [3, 1, 0, 2]]],
[[1, [0, 1, 3, 2]], [1, [1, 0, 2, 3]], [1, [2, 3, 0, 1]], [1, [3, 2, 1, 0]]],
[[1, [0, 1, 2, 3]], [1, [1, 0, 3, 2]], [1, [2, 3, 1, 0]], [1, [3, 2, 0, 1]]],
[[1, [0, 3, 2, 1]], [1, [1, 0, 3, 2]], [1, [2, 1, 0, 3]], [1, [3, 2, 1, 0]]],
[[1, [0, 3, 1, 2]], [1, [1, 0, 2, 3]], [1, [2, 1, 3, 0]], [1, [3, 2, 0, 1]]],
[[1, [0, 1, 3, 2]], [1, [1, 3, 2, 0]], [1, [2, 0, 1, 3]], [1, [3, 2, 0, 1]]],
[[1, [0, 2, 3, 1]], [1, [1, 3, 0, 2]], [1, [2, 0, 1, 3]], [1, [3, 1, 2, 0]]],
[[1, [0, 3, 1, 2]], [1, [1, 2, 0, 3]], [1, [2, 0, 3, 1]], [1, [3, 1, 2, 0]]],
[[1, [0, 3, 2, 1]], [1, [1, 2, 3, 0]], [1, [2, 0, 1, 3]], [1, [3, 1, 0, 2]]],
[[1, [0, 1, 3, 2]], [1, [1, 2, 0, 3]], [1, [2, 3, 1, 0]], [1, [3, 0, 2, 1]]],
[[1, [0, 1, 2, 3]], [1, [1, 2, 3, 0]], [1, [2, 3, 0, 1]], [1, [3, 0, 1, 2]]],
[[1, [0, 3, 2, 1]], [1, [1, 2, 0, 3]], [1, [2, 1, 3, 0]], [1, [3, 0, 1, 2]]],
[[6, [0, 1]], [4, [2, 1, 0]], [-5, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [-3, [1, 0, 3, 2]]],
[[-4, [0, 2, 1]], [1, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [6, [0, 2, 3, 1]], [6, [0, 3, 1, 2]], [3, [1, 0, 2, 3]]],
[[-4, [2, 0, 1]], [3, [2, 3, 1, 0]], [6, [3, 0, 2, 1]], [6, [3, 1, 0, 2]], [-4, [3, 1, 2, 0]], [1, [3, 2, 1, 0]]]]

dim_20_n_at_most_4_non_square = n_at_most_4_non_square[:-3]
dim_21_n_at_most_4_non_square = n_at_most_4_non_square[:-2]

It appears there are $(n-2)^2$ new non-latin constant covers added to the dimension at each step. What are these covers?

In [5]:
# n = 3
n_is_3_non_latins = [
    #Non-vanishing
    [[-3, [0, 1]], [2, [0, 1, 2]], [-2, [2, 1, 0]]]
    ]
# n = 4
n_is_4_non_latins = [
    #Non-vanishing
    [[2, [0, 1, 2]], [2, [2, 1, 0]], [3, [1, 3, 0, 2]], [3, [2, 0, 3, 1]]],
    [[6, [0, 1]], [4, [2, 1, 0]], [-5, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [-3, [1, 0, 3, 2]]],
    #Vanishing, D4 symmetrical
    [[-4, [0, 2, 1]], [1, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [6, [0, 2, 3, 1]], [6, [0, 3, 1, 2]], [3, [1, 0, 2, 3]]],
    [[-4, [2, 0, 1]], [3, [2, 3, 1, 0]], [6, [3, 0, 2, 1]], [6, [3, 1, 0, 2]], [-4, [3, 1, 2, 0]], [1, [3, 2, 1, 0]]]
    ]
# n = 5
n_is_5_non_latins = [
    #Non-vanishing, These six are in two D4 equivalence classes.
    [[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 3, 2, 1]], [36, [1, 4, 3, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 2, 4, 3, 1]], [-36, [1, 2, 4, 3, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[-36, [5, 4, 2, 3, 1]], [36, [1, 4, 2, 3, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[-36, [5, 3, 4, 2, 1]], [36, [1, 3, 4, 2, 5]], [15, [2, 1, 4, 3]], [10, [3, 2, 1]]],
    [[36, [5, 3, 2, 4, 1]], [-36, [1, 3, 2, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    
    #No D4 symmetries
    [[36, [4, 1, 3, 5, 2]], [-36, [2, 5, 3, 1, 4]], [30, [2, 4, 1, 3]], [20, [1, 2, 3]], [15, [2, 1]]],
    [[12, [5, 4, 3, 2, 1]], [-12, [1, 2, 3, 4, 5]], [15, [1, 2, 3, 4]], [-10, [1, 2, 3]], [5, [1, 2]]],
    [[-24, [4, 5, 3, 1, 2]], [24, [2, 1, 3, 5, 4]], [15, [3, 2, 1, 4]], [15, [1, 4, 3, 2]], [5, [2, 1]]],
    
]
# n = 6
n_is_6_non_latins = [
    #Vanishing
    
    #Unique, 7 terms
    [[1, [4, 3, 2, 1, 0]], [-5/6, [4, 5, 3, 2, 1, 0]], [-4/3, [5, 3, 4, 2, 1, 0]], [-3/2, [5, 4, 2, 3, 1, 0]], [-4/3, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [5/6, [5, 4, 3, 2, 1, 0]]],
    
    #D4 equivalent, 8 terms
    [[1, [4, 2, 3, 1, 0]], [-5/6, [4, 5, 3, 2, 1, 0]], [-2, [5, 2, 4, 3, 1, 0]], [-2, [5, 3, 2, 4, 1, 0]], [1, [5, 4, 2, 3, 1, 0]], [-4/3, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [1, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 3, 1, 2, 0]], [-5/6, [4, 5, 3, 2, 1, 0]], [-4/3, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 1, 3, 2, 0]], [-2, [5, 4, 2, 1, 3, 0]], [1, [5, 4, 2, 3, 1, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [1, [5, 4, 3, 2, 1, 0]]],
    
    #D4 equivalent, 9 terms
    [[1, [3, 4, 2, 1, 0]], [-5/3, [3, 5, 4, 2, 1, 0]], [-5/3, [4, 3, 5, 2, 1, 0]], [-5/3, [4, 5, 3, 2, 1, 0]], [4/3, [5, 3, 4, 2, 1, 0]], [-3/2, [5, 4, 2, 3, 1, 0]], [-4/3, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [7/3, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 3, 2, 0, 1]], [-5/6, [4, 5, 3, 2, 1, 0]], [-4/3, [5, 3, 4, 2, 1, 0]], [-3/2, [5, 4, 2, 3, 1, 0]], [-5/3, [5, 4, 3, 0, 2, 1]], [-5/3, [5, 4, 3, 1, 0, 2]], [4/3, [5, 4, 3, 1, 2, 0]], [-5/3, [5, 4, 3, 2, 0, 1]], [7/3, [5, 4, 3, 2, 1, 0]]],
    
    
    #Two D4 equivalence classes, 12 terms
    [[1, [2, 4, 3, 1, 0]], [-5/2, [2, 5, 4, 3, 1, 0]], [-5/2, [3, 5, 4, 2, 1, 0]], [-5/3, [4, 3, 5, 2, 1, 0]], [5/3, [4, 5, 3, 2, 1, 0]], [-1/2, [5, 2, 4, 3, 1, 0]], [-2, [5, 3, 2, 4, 1, 0]], [19/6, [5, 3, 4, 2, 1, 0]], [2, [5, 4, 2, 3, 1, 0]], [-4/3, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-1/2, [5, 4, 3, 2, 1, 0]]],
    [[1, [3, 2, 4, 1, 0]], [-5/3, [3, 5, 4, 2, 1, 0]], [-5/2, [4, 3, 2, 5, 1, 0]], [-5/2, [4, 3, 5, 2, 1, 0]], [5/3, [4, 5, 3, 2, 1, 0]], [-2, [5, 2, 4, 3, 1, 0]], [-1/2, [5, 3, 2, 4, 1, 0]], [19/6, [5, 3, 4, 2, 1, 0]], [2, [5, 4, 2, 3, 1, 0]], [-4/3, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-1/2, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 1, 3, 2, 0]], [-5/6, [4, 5, 3, 2, 1, 0]], [-8/3, [5, 1, 4, 3, 2, 0]], [-4/3, [5, 2, 4, 3, 1, 0]], [-2, [5, 3, 2, 4, 1, 0]], [2, [5, 3, 4, 2, 1, 0]], [-4/3, [5, 4, 1, 3, 2, 0]], [-2, [5, 4, 2, 1, 3, 0]], [10/3, [5, 4, 2, 3, 1, 0]], [2, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-4/3, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 2, 1, 3, 0]], [-5/6, [4, 5, 3, 2, 1, 0]], [-2, [5, 2, 4, 3, 1, 0]], [-8/3, [5, 3, 2, 1, 4, 0]], [-4/3, [5, 3, 2, 4, 1, 0]], [2, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 1, 3, 2, 0]], [-4/3, [5, 4, 2, 1, 3, 0]], [10/3, [5, 4, 2, 3, 1, 0]], [2, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-4/3, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 3, 0, 2, 1]], [-5/6, [4, 5, 3, 2, 1, 0]], [-4/3, [5, 3, 4, 2, 1, 0]], [-5/2, [5, 4, 0, 3, 2, 1]], [-1/2, [5, 4, 1, 3, 2, 0]], [-2, [5, 4, 2, 1, 3, 0]], [2, [5, 4, 2, 3, 1, 0]], [-5/2, [5, 4, 3, 0, 2, 1]], [-5/3, [5, 4, 3, 1, 0, 2]], [19/6, [5, 4, 3, 1, 2, 0]], [5/3, [5, 4, 3, 2, 0, 1]], [-1/2, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 3, 1, 0, 2]], [-5/6, [4, 5, 3, 2, 1, 0]], [-4/3, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 1, 3, 2, 0]], [-5/2, [5, 4, 2, 1, 0, 3]], [-1/2, [5, 4, 2, 1, 3, 0]], [2, [5, 4, 2, 3, 1, 0]], [-5/3, [5, 4, 3, 0, 2, 1]], [-5/2, [5, 4, 3, 1, 0, 2]], [19/6, [5, 4, 3, 1, 2, 0]], [5/3, [5, 4, 3, 2, 0, 1]], [-1/2, [5, 4, 3, 2, 1, 0]]],
    
    #D4 equivalent, 14 terms
    [[1, [1, 4, 3, 2, 0]], [-10/3, [1, 5, 4, 3, 2, 0]], [-5/3, [2, 5, 4, 3, 1, 0]], [-5/3, [4, 3, 5, 2, 1, 0]], [5/3, [4, 5, 3, 2, 1, 0]], [-2/3, [5, 1, 4, 3, 2, 0]], [-1/3, [5, 2, 4, 3, 1, 0]], [-2, [5, 3, 2, 4, 1, 0]], [11/3, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 2, 1, 3, 0]], [4, [5, 4, 2, 3, 1, 0]], [2, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-23/6, [5, 4, 3, 2, 1, 0]]],
    [[1, [3, 2, 1, 4, 0]], [-5/3, [3, 5, 4, 2, 1, 0]], [-10/3, [4, 3, 2, 1, 5, 0]], [-5/3, [4, 3, 2, 5, 1, 0]], [5/3, [4, 5, 3, 2, 1, 0]], [-2, [5, 2, 4, 3, 1, 0]], [-2/3, [5, 3, 2, 1, 4, 0]], [-1/3, [5, 3, 2, 4, 1, 0]], [11/3, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 1, 3, 2, 0]], [4, [5, 4, 2, 3, 1, 0]], [2, [5, 4, 3, 1, 2, 0]], [-5/6, [5, 4, 3, 2, 0, 1]], [-23/6, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 0, 3, 2, 1]], [-5/6, [4, 5, 3, 2, 1, 0]], [-10/3, [5, 0, 4, 3, 2, 1]], [-2/3, [5, 1, 4, 3, 2, 0]], [-2, [5, 3, 2, 4, 1, 0]], [2, [5, 3, 4, 2, 1, 0]], [-5/3, [5, 4, 0, 3, 2, 1]], [-1/3, [5, 4, 1, 3, 2, 0]], [-2, [5, 4, 2, 1, 3, 0]], [4, [5, 4, 2, 3, 1, 0]], [-5/3, [5, 4, 3, 1, 0, 2]], [11/3, [5, 4, 3, 1, 2, 0]], [5/3, [5, 4, 3, 2, 0, 1]], [-23/6, [5, 4, 3, 2, 1, 0]]],
    [[1, [4, 2, 1, 0, 3]], [-5/6, [4, 5, 3, 2, 1, 0]], [-2, [5, 2, 4, 3, 1, 0]], [-10/3, [5, 3, 2, 1, 0, 4]], [-2/3, [5, 3, 2, 1, 4, 0]], [2, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 1, 3, 2, 0]], [-5/3, [5, 4, 2, 1, 0, 3]], [-1/3, [5, 4, 2, 1, 3, 0]], [4, [5, 4, 2, 3, 1, 0]], [-5/3, [5, 4, 3, 0, 2, 1]], [11/3, [5, 4, 3, 1, 2, 0]], [5/3, [5, 4, 3, 2, 0, 1]], [-23/6, [5, 4, 3, 2, 1, 0]]],
    
    #Unique, 15 terms
    [[1, [0, 4, 3, 2, 1]], [-25/6, [0, 5, 4, 3, 2, 1]], [-5/6, [1, 5, 4, 3, 2, 0]], [-5/3, [4, 3, 5, 2, 1, 0]], [5/3, [4, 5, 3, 2, 1, 0]], [-5/6, [5, 0, 4, 3, 2, 1]], [-1/6, [5, 1, 4, 3, 2, 0]], [-2, [5, 3, 2, 4, 1, 0]], [11/3, [5, 3, 4, 2, 1, 0]], [-2, [5, 4, 2, 1, 3, 0]], [4, [5, 4, 2, 3, 1, 0]], [-5/3, [5, 4, 3, 1, 0, 2]], [11/3, [5, 4, 3, 1, 2, 0]], [5/3, [5, 4, 3, 2, 0, 1]], [-19/3, [5, 4, 3, 2, 1, 0]]]
]

In [3]:
start_with_5_latins = [
    [[1,[1]]],
    [[1, [2, 1]], [1, [1, 2]]],
    [[1, [3, 2, 1]], [1, [2, 1, 3]], [1, [1, 3, 2]]],
    [[1, [3, 1, 2]], [1, [2, 3, 1]], [1, [1, 2, 3]]],
    [[-2, [3, 2, 1]], [2, [1, 2, 3]], [-3, [1, 2]]],
    [[1, [4, 2, 3, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 3, 2, 4]]],
    [[1, [4, 2, 1, 3]], [1, [3, 4, 2, 1]], [1, [2, 1, 3, 4]], [1, [1, 3, 4, 2]]],
    [[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 2, 1]], [1, [3, 4, 1, 2]], [1, [2, 1, 3, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 3, 1, 2]], [1, [3, 4, 2, 1]], [1, [2, 1, 4, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 2, 1]], [1, [3, 2, 1, 4]], [1, [2, 1, 4, 3]], [1, [1, 4, 3, 2]]],
    [[1, [4, 3, 1, 2]], [1, [3, 2, 4, 1]], [1, [2, 1, 3, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 3, 2, 1]], [1, [3, 1, 4, 2]], [1, [2, 4, 1, 3]], [1, [1, 2, 3, 4]]],
    [[1, [4, 3, 1, 2]], [1, [3, 1, 2, 4]], [1, [2, 4, 3, 1]], [1, [1, 2, 4, 3]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 2, 4]], [1, [2, 4, 1, 3]], [1, [1, 3, 4, 2]]],
    [[1, [4, 2, 3, 1]], [1, [3, 1, 4, 2]], [1, [2, 3, 1, 4]], [1, [1, 4, 2, 3]]],
    [[1, [4, 2, 1, 3]], [1, [3, 1, 2, 4]], [1, [2, 3, 4, 1]], [1, [1, 4, 3, 2]]],
    [[1, [4, 1, 3, 2]], [1, [3, 4, 2, 1]], [1, [2, 3, 1, 4]], [1, [1, 2, 4, 3]]],
    [[1, [4, 1, 2, 3]], [1, [3, 4, 1, 2]], [1, [2, 3, 4, 1]], [1, [1, 2, 3, 4]]],
    [[1, [4, 1, 2, 3]], [1, [3, 2, 4, 1]], [1, [2, 3, 1, 4]], [1, [1, 4, 3, 2]]],
    [[3, [3, 1, 4, 2]], [3, [2, 4, 1, 3]], [2, [3, 2, 1]], [2, [1, 2, 3]]],
    [[-3, [2, 1, 4, 3]], [-4, [1, 3, 2, 4]], [-5, [1, 2, 3, 4]], [4, [3, 2, 1]], [6, [1, 2]]],
    [[-4, [0, 2, 1]], [1, [0, 1, 2, 3]], [-4, [0, 2, 1, 3]], [6, [0, 2, 3, 1]], [6, [0, 3, 1, 2]], [3, [1, 0, 2, 3]]],
    [[-4, [2, 0, 1]], [3, [2, 3, 1, 0]], [6, [3, 0, 2, 1]], [6, [3, 1, 0, 2]], [-4, [3, 1, 2, 0]], [1, [3, 2, 1, 0]]],
    
    [[1, [0, 1, 2, 3, 4]], [1, [1, 2, 3, 4, 0]], [1, [2, 3, 4, 0, 1]], [1, [3, 4, 0, 1, 2]], [1, [4, 0, 1, 2, 3]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 3, 0, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 2, 1, 0]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 0, 2, 4, 3]], [1, [2, 4, 3, 1, 0]], [1, [3, 2, 1, 0, 4]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 3, 4, 0]], [1, [2, 4, 1, 0, 3]], [1, [3, 0, 2, 1, 4]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 2, 3, 4, 1]], [1, [1, 3, 4, 2, 0]], [1, [2, 4, 1, 0, 3]], [1, [3, 0, 2, 1, 4]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 3, 4, 0, 2]], [1, [2, 1, 3, 4, 0]], [1, [3, 0, 2, 1, 4]], [1, [4, 2, 0, 3, 1]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 3, 4, 0, 2]], [1, [2, 1, 3, 4, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 0, 2, 3, 1]]],
    [[1, [0, 1, 3, 4, 2]], [1, [1, 0, 2, 3, 4]], [1, [2, 4, 0, 1, 3]], [1, [3, 2, 4, 0, 1]], [1, [4, 3, 1, 2, 0]]],
    [[1, [0, 1, 4, 2, 3]], [1, [1, 4, 0, 3, 2]], [1, [2, 3, 1, 4, 0]], [1, [3, 0, 2, 1, 4]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 1, 4, 2, 3]], [1, [1, 0, 2, 3, 4]], [1, [2, 3, 1, 4, 0]], [1, [3, 4, 0, 1, 2]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 4, 3, 2, 1]], [1, [1, 0, 4, 3, 2]], [1, [2, 3, 1, 4, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 3, 0, 2, 4]], [1, [2, 1, 3, 4, 0]], [1, [3, 0, 4, 1, 2]], [1, [4, 2, 1, 0, 3]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 0, 3, 2, 4]], [1, [2, 3, 4, 1, 0]], [1, [3, 1, 0, 4, 2]], [1, [4, 2, 1, 0, 3]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 0, 4, 2, 3]], [1, [2, 1, 3, 4, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 3, 1, 0, 2]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 0, 4, 2, 3]], [1, [2, 1, 3, 4, 0]], [1, [3, 4, 1, 0, 2]], [1, [4, 2, 0, 3, 1]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 0, 4, 3, 2]], [1, [2, 4, 1, 0, 3]], [1, [3, 2, 0, 4, 1]], [1, [4, 1, 3, 2, 0]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 4, 3, 0, 2]], [1, [2, 0, 4, 3, 1]], [1, [3, 2, 1, 4, 0]], [1, [4, 1, 0, 2, 3]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 4, 3, 0, 2]], [1, [2, 1, 0, 3, 4]], [1, [3, 2, 1, 4, 0]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 1, 4, 2, 3]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 0, 1, 4]], [1, [3, 2, 1, 4, 0]], [1, [4, 0, 2, 3, 1]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 0, 1, 4]], [1, [3, 1, 4, 2, 0]], [1, [4, 0, 2, 3, 1]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 0, 1, 4]], [1, [3, 0, 4, 2, 1]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 1, 2, 4, 3]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 0, 1, 4]], [1, [3, 0, 4, 2, 1]], [1, [4, 2, 1, 3, 0]]],
    [[1, [0, 4, 3, 2, 1]], [1, [1, 2, 4, 0, 3]], [1, [2, 3, 0, 1, 4]], [1, [3, 0, 1, 4, 2]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 2, 3, 0, 4]], [1, [2, 3, 0, 4, 1]], [1, [3, 0, 4, 1, 2]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 3, 0, 4]], [1, [2, 4, 0, 1, 3]], [1, [3, 0, 1, 4, 2]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 3, 0, 4]], [1, [2, 4, 0, 1, 3]], [1, [3, 1, 2, 4, 0]], [1, [4, 0, 1, 3, 2]]],
    [[1, [0, 3, 4, 1, 2]], [1, [1, 2, 3, 0, 4]], [1, [2, 4, 0, 3, 1]], [1, [3, 1, 2, 4, 0]], [1, [4, 0, 1, 2, 3]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 3, 2, 4, 0]], [1, [2, 0, 4, 1, 3]], [1, [3, 1, 0, 2, 4]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 3, 1, 2, 4]], [1, [1, 4, 2, 3, 0]], [1, [2, 0, 4, 1, 3]], [1, [3, 1, 0, 4, 2]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 3, 1, 2, 4]], [1, [1, 4, 0, 3, 2]], [1, [2, 0, 4, 1, 3]], [1, [3, 1, 2, 4, 0]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 3, 0, 2, 4]], [1, [2, 0, 4, 1, 3]], [1, [3, 1, 2, 4, 0]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 2, 4, 0, 3]], [1, [2, 0, 3, 4, 1]], [1, [3, 1, 0, 2, 4]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 2, 0, 3, 4]], [1, [2, 0, 3, 4, 1]], [1, [3, 1, 4, 0, 2]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 3, 4, 0, 2]], [1, [2, 0, 3, 4, 1]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 3, 4, 0, 2]], [1, [2, 0, 1, 4, 3]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 3, 2, 0]]],
    [[1, [0, 1, 3, 4, 2]], [1, [1, 4, 2, 3, 0]], [1, [2, 3, 4, 0, 1]], [1, [3, 2, 0, 1, 4]], [1, [4, 0, 1, 2, 3]]],
    [[1, [0, 4, 3, 1, 2]], [1, [1, 0, 2, 4, 3]], [1, [2, 1, 0, 3, 4]], [1, [3, 2, 4, 0, 1]], [1, [4, 3, 1, 2, 0]]],
    [[1, [0, 2, 4, 1, 3]], [1, [1, 4, 3, 0, 2]], [1, [2, 1, 0, 3, 4]], [1, [3, 0, 2, 4, 1]], [1, [4, 3, 1, 2, 0]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 4, 3, 0, 2]], [1, [2, 1, 0, 4, 3]], [1, [3, 0, 4, 2, 1]], [1, [4, 2, 1, 3, 0]]],
    [[1, [0, 2, 1, 3, 4]], [1, [1, 4, 3, 0, 2]], [1, [2, 1, 0, 4, 3]], [1, [3, 0, 4, 2, 1]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 2, 4, 3, 1]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 0, 1, 4]], [1, [3, 1, 2, 4, 0]], [1, [4, 0, 1, 2, 3]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 0, 4, 3]], [1, [2, 0, 1, 3, 4]], [1, [3, 4, 2, 1, 0]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 0, 3, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 2, 1, 0]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 3, 2, 4, 1]], [1, [1, 0, 4, 2, 3]], [1, [2, 4, 1, 3, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 0, 4, 2, 3]], [1, [2, 3, 1, 4, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 0, 4, 2, 3]], [1, [2, 3, 1, 4, 0]], [1, [3, 4, 0, 1, 2]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 4, 0, 2, 3]], [1, [2, 3, 4, 1, 0]], [1, [3, 0, 1, 4, 2]], [1, [4, 2, 3, 0, 1]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 3, 4, 2, 0]], [1, [2, 4, 3, 0, 1]], [1, [3, 0, 1, 4, 2]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 0, 4, 2, 3]], [1, [2, 3, 1, 0, 4]], [1, [3, 1, 0, 4, 2]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 4, 2, 1, 3]], [1, [1, 0, 3, 2, 4]], [1, [2, 3, 4, 0, 1]], [1, [3, 1, 0, 4, 2]], [1, [4, 2, 1, 3, 0]]],
    [[1, [0, 4, 2, 1, 3]], [1, [1, 0, 3, 2, 4]], [1, [2, 3, 4, 0, 1]], [1, [3, 2, 1, 4, 0]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 4, 2, 1, 3]], [1, [1, 2, 3, 4, 0]], [1, [2, 3, 4, 0, 1]], [1, [3, 0, 1, 2, 4]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 1, 3, 4, 2]], [1, [1, 3, 4, 2, 0]], [1, [2, 4, 0, 3, 1]], [1, [3, 2, 1, 0, 4]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 4, 2, 3, 1]], [1, [1, 3, 4, 2, 0]], [1, [2, 1, 0, 4, 3]], [1, [3, 2, 1, 0, 4]], [1, [4, 0, 3, 1, 2]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 2, 0, 4, 3]], [1, [2, 0, 4, 3, 1]], [1, [3, 4, 1, 2, 0]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 2, 0, 4, 3]], [1, [2, 4, 1, 3, 0]], [1, [3, 0, 4, 2, 1]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 2, 4, 3, 1]], [1, [1, 4, 0, 2, 3]], [1, [2, 3, 1, 4, 0]], [1, [3, 0, 2, 1, 4]], [1, [4, 1, 3, 0, 2]]],
    [[1, [0, 1, 3, 4, 2]], [1, [1, 2, 4, 3, 0]], [1, [2, 3, 1, 0, 4]], [1, [3, 4, 0, 2, 1]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 4, 0, 2, 3]], [1, [2, 3, 1, 4, 0]], [1, [3, 2, 4, 0, 1]], [1, [4, 0, 3, 1, 2]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 4, 0, 2, 3]], [1, [2, 3, 4, 0, 1]], [1, [3, 0, 1, 4, 2]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 2, 3, 1, 4]], [1, [1, 4, 0, 2, 3]], [1, [2, 3, 4, 0, 1]], [1, [3, 0, 1, 4, 2]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 4, 2, 0, 3]], [1, [2, 1, 3, 4, 0]], [1, [3, 2, 0, 1, 4]], [1, [4, 0, 1, 3, 2]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 2, 4, 0, 3]], [1, [2, 1, 3, 4, 0]], [1, [3, 0, 2, 1, 4]], [1, [4, 3, 0, 2, 1]]],
    [[1, [0, 4, 3, 1, 2]], [1, [1, 2, 0, 3, 4]], [1, [2, 3, 1, 4, 0]], [1, [3, 0, 4, 2, 1]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 3, 2, 4, 1]], [1, [1, 4, 0, 3, 2]], [1, [2, 0, 3, 1, 4]], [1, [3, 1, 4, 2, 0]], [1, [4, 2, 1, 0, 3]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 4, 0, 3, 2]], [1, [2, 3, 4, 0, 1]], [1, [3, 0, 2, 1, 4]], [1, [4, 1, 3, 2, 0]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 4, 0, 3, 2]], [1, [2, 0, 1, 4, 3]], [1, [3, 2, 4, 0, 1]], [1, [4, 1, 3, 2, 0]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 4, 3, 2, 0]], [1, [2, 0, 1, 4, 3]], [1, [3, 2, 4, 0, 1]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 3, 2, 1, 4]], [1, [1, 4, 3, 2, 0]], [1, [2, 0, 1, 4, 3]], [1, [3, 1, 4, 0, 2]], [1, [4, 2, 0, 3, 1]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 0, 3, 4, 2]], [1, [2, 1, 0, 3, 4]], [1, [3, 2, 4, 0, 1]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 3, 4, 1, 2]], [1, [1, 2, 0, 3, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 2, 0, 1]], [1, [4, 1, 3, 2, 0]]],
    [[1, [0, 3, 4, 1, 2]], [1, [1, 2, 3, 0, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 0, 2, 1]], [1, [4, 1, 2, 3, 0]]],
    [[1, [0, 3, 1, 4, 2]], [1, [1, 2, 4, 3, 0]], [1, [2, 0, 3, 1, 4]], [1, [3, 4, 0, 2, 1]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 3, 1, 4, 2]], [1, [1, 4, 3, 2, 0]], [1, [2, 0, 4, 3, 1]], [1, [3, 2, 0, 1, 4]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 3, 1, 4, 2]], [1, [1, 4, 0, 2, 3]], [1, [2, 0, 4, 3, 1]], [1, [3, 1, 2, 0, 4]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 3, 1, 4, 2]], [1, [1, 4, 3, 2, 0]], [1, [2, 0, 4, 3, 1]], [1, [3, 1, 2, 0, 4]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 1, 2, 3, 4]], [1, [1, 4, 3, 0, 2]], [1, [2, 3, 1, 4, 0]], [1, [3, 0, 4, 2, 1]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 3, 2, 4, 0]], [1, [2, 1, 3, 0, 4]], [1, [3, 0, 4, 2, 1]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 3, 2, 4, 1]], [1, [1, 0, 4, 3, 2]], [1, [2, 1, 3, 0, 4]], [1, [3, 4, 1, 2, 0]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 0, 4, 3, 2]], [1, [2, 1, 3, 0, 4]], [1, [3, 4, 2, 1, 0]], [1, [4, 3, 0, 2, 1]]],
    [[1, [0, 2, 4, 1, 3]], [1, [1, 0, 3, 4, 2]], [1, [2, 1, 0, 3, 4]], [1, [3, 4, 1, 2, 0]], [1, [4, 3, 2, 0, 1]]],
    [[1, [0, 2, 3, 1, 4]], [1, [1, 0, 4, 3, 2]], [1, [2, 3, 0, 4, 1]], [1, [3, 4, 1, 2, 0]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 2, 3, 1, 4]], [1, [1, 3, 0, 4, 2]], [1, [2, 1, 4, 0, 3]], [1, [3, 4, 1, 2, 0]], [1, [4, 0, 2, 3, 1]]],
    [[1, [0, 2, 4, 3, 1]], [1, [1, 3, 0, 4, 2]], [1, [2, 1, 3, 0, 4]], [1, [3, 4, 1, 2, 0]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 2, 4, 3, 1]], [1, [1, 3, 0, 4, 2]], [1, [2, 4, 3, 1, 0]], [1, [3, 0, 1, 2, 4]], [1, [4, 1, 2, 0, 3]]],
    [[1, [0, 1, 4, 3, 2]], [1, [1, 2, 3, 4, 0]], [1, [2, 3, 1, 0, 4]], [1, [3, 4, 0, 2, 1]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 3, 4, 0]], [1, [2, 4, 0, 1, 3]], [1, [3, 1, 2, 0, 4]], [1, [4, 0, 1, 3, 2]]],
    [[1, [0, 3, 4, 2, 1]], [1, [1, 2, 0, 4, 3]], [1, [2, 4, 3, 1, 0]], [1, [3, 1, 2, 0, 4]], [1, [4, 0, 1, 3, 2]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 0, 4, 3, 2]], [1, [2, 4, 3, 1, 0]], [1, [3, 1, 2, 0, 4]], [1, [4, 3, 0, 2, 1]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 0, 4, 3, 2]], [1, [2, 4, 3, 1, 0]], [1, [3, 1, 0, 2, 4]], [1, [4, 3, 2, 0, 1]]],
    [[1, [0, 2, 3, 4, 1]], [1, [1, 0, 4, 3, 2]], [1, [2, 4, 1, 0, 3]], [1, [3, 1, 0, 2, 4]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 2, 3, 4, 1]], [1, [1, 0, 4, 2, 3]], [1, [2, 4, 1, 3, 0]], [1, [3, 1, 2, 0, 4]], [1, [4, 3, 0, 1, 2]]],
    [[1, [0, 1, 4, 3, 2]], [1, [1, 3, 0, 2, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 2, 0, 1]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 1, 4, 3, 2]], [1, [1, 3, 2, 0, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 4, 0, 2, 1]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 2, 1, 4, 3]], [1, [1, 4, 2, 3, 0]], [1, [2, 3, 0, 1, 4]], [1, [3, 1, 4, 0, 2]], [1, [4, 0, 3, 2, 1]]],
    [[1, [0, 4, 1, 2, 3]], [1, [1, 0, 2, 3, 4]], [1, [2, 3, 0, 4, 1]], [1, [3, 1, 4, 0, 2]], [1, [4, 2, 3, 1, 0]]],
    [[1, [0, 4, 3, 2, 1]], [1, [1, 0, 2, 3, 4]], [1, [2, 3, 1, 4, 0]], [1, [3, 1, 4, 0, 2]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 1, 3, 2, 4]], [1, [1, 0, 4, 3, 2]], [1, [2, 3, 1, 4, 0]], [1, [3, 4, 2, 0, 1]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 1, 3, 2, 4]], [1, [1, 3, 4, 0, 2]], [1, [2, 4, 1, 3, 0]], [1, [3, 0, 2, 4, 1]], [1, [4, 2, 0, 1, 3]]],
    [[1, [0, 1, 3, 2, 4]], [1, [1, 0, 4, 3, 2]], [1, [2, 4, 1, 0, 3]], [1, [3, 2, 0, 4, 1]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 1, 4, 3, 2]], [1, [1, 0, 3, 2, 4]], [1, [2, 4, 1, 0, 3]], [1, [3, 2, 0, 4, 1]], [1, [4, 3, 2, 1, 0]]],
    [[1, [0, 4, 1, 3, 2]], [1, [1, 3, 4, 2, 0]], [1, [2, 1, 3, 0, 4]], [1, [3, 2, 0, 4, 1]], [1, [4, 0, 2, 1, 3]]],
    [[1, [0, 4, 3, 2, 1]], [1, [1, 3, 2, 0, 4]], [1, [2, 0, 1, 4, 3]], [1, [3, 2, 4, 1, 0]], [1, [4, 1, 0, 3, 2]]],
    [[1, [0, 4, 2, 1, 3]], [1, [1, 2, 3, 0, 4]], [1, [2, 1, 4, 3, 0]], [1, [3, 0, 1, 4, 2]], [1, [4, 3, 0, 2, 1]]],
    
    [[36, [5, 2, 3, 4, 1]], [-36, [1, 2, 3, 4, 5]], [15, [3, 4, 1, 2]], [10, [1, 2, 3]]],
    [[36, [4, 1, 3, 5, 2]], [-36, [2, 5, 3, 1, 4]], [30, [2, 4, 1, 3]], [20, [1, 2, 3]], [15, [2, 1]]],
    [[12, [5, 4, 3, 2, 1]], [-12, [1, 2, 3, 4, 5]], [15, [1, 2, 3, 4]], [-10, [1, 2, 3]], [5, [1, 2]]],
    [[-24, [4, 5, 3, 1, 2]], [24, [2, 1, 3, 5, 4]], [15, [3, 2, 1, 4]], [15, [1, 4, 3, 2]], [5, [2, 1]]],
    
    [[-4, [4, 3, 2, 0, 1]], [-6, [4, 3, 1, 2, 0]], [-6, [4, 2, 3, 1, 0]], [-4, [3, 4, 2, 1, 0]], [5, [3, 2, 1, 0]]],
    [[4, [4, 3, 2, 1, 0]], [-4, [4, 3, 2, 0, 1]], [6, [4, 3, 1, 2, 0]], [-8, [4, 3, 1, 0, 2]], [-8, [4, 3, 0, 2, 1]], [-6, [4, 2, 3, 1, 0]], [-4, [3, 4, 2, 1, 0]], [5, [3, 2, 0, 1]]],
    [[-8, [4, 3, 2, 1, 0]], [-4, [4, 3, 2, 0, 1]], [9, [4, 3, 1, 2, 0]], [15, [4, 2, 3, 1, 0]], [-9, [4, 2, 1, 3, 0]], [-3, [4, 1, 3, 2, 0]], [8, [3, 4, 2, 1, 0]], [-8, [3, 2, 4, 1, 0]], [-8, [2, 4, 3, 1, 0]], [-12, [1, 4, 3, 2, 0]], [5, [1, 3, 2, 0]]],
    [[-8, [4, 3, 2, 1, 0]], [-4, [4, 3, 2, 0, 1]], [9, [4, 3, 1, 2, 0]], [15, [4, 2, 3, 1, 0]], [-3, [4, 2, 1, 3, 0]], [-9, [4, 1, 3, 2, 0]], [8, [3, 4, 2, 1, 0]], [-8, [3, 2, 4, 1, 0]], [-12, [3, 2, 1, 4, 0]], [-8, [2, 4, 3, 1, 0]], [5, [2, 1, 3, 0]]],
    [[-20, [4, 3, 2, 1, 0]], [8, [4, 3, 2, 0, 1]], [17, [4, 3, 1, 2, 0]], [-8, [4, 3, 1, 0, 2]], [17, [4, 2, 3, 1, 0]], [-9, [4, 2, 1, 3, 0]], [-1, [4, 1, 3, 2, 0]], [-4, [4, 0, 3, 2, 1]], [8, [3, 4, 2, 1, 0]], [-8, [3, 2, 4, 1, 0]], [-4, [1, 4, 3, 2, 0]], [-16, [0, 4, 3, 2, 1]], [5, [0, 3, 2, 1]]],

    

]